# Adrianic v4.8 — Ambiguous Topology + Endogenous Structural Stopping

Adds branching, cycles, shortcuts, convergent decoys and full-length wrong paths. Then removes the supplied traversal depth and asks the system to choose where to stop from structural evidence alone.

In [ ]:
from __future__ import annotations
from pathlib import Path
from collections import Counter, defaultdict
import argparse, copy, json, math, random, re, urllib.request, hashlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# -----------------------------------------------------------------------------
# Adrianic v4.4 — Multi-consequence value + adaptive-gate binding test:
#   (1) pairwise + latent common-cause evidence control (v3.21 family)
#   (2) comparative replacement + honest address/template separation (field gate)
#
# Scientific boundary: this is a tiny recurrent LM + explicit memory controller.
# It tests functional relational binding under interference, not human understanding.
# -----------------------------------------------------------------------------

if Path('/content').exists(): ROOT = Path('/content')
elif Path('/mnt/data').exists(): ROOT = Path('/mnt/data')
else: ROOT = Path.cwd()
OUT = ROOT / 'adrianic_v44_multiconsequence_results'
OUT.mkdir(parents=True, exist_ok=True)

TOKEN_RE = re.compile(r"[a-zA-Z]+|[0-9]+|[^\w\s]")
STOP = set('the a an is are in of to and this that with for near under before after through beyond fact remember'.split())

def seed_all(seed:int):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def tokenize(s:str): return TOKEN_RE.findall(s.lower())

def content_tokens(s:str):
    return [w for w in tokenize(s) if w.isalpha() and w not in STOP]

FACTS=[
('two plus two equals','four'),('three plus five equals','eight'),('seven minus three equals','four'),('six times seven equals','fortytwo'),
('the square root of nine is','three'),('a triangle has','three'),('a square has','four'),('the additive identity is','zero'),
('the si unit of force is the','newton'),('the si unit of energy is the','joule'),('the si unit of power is the','watt'),('mass times acceleration gives','force'),
('energy associated with motion is','kinetic'),('the force attracting masses is','gravity'),('the chemical symbol for hydrogen is','h'),
('the chemical symbol for oxygen is','o'),('water contains hydrogen and','oxygen'),('an atom with six protons is','carbon'),
('an atom with eight protons is','oxygen'),('the center of an atom is the','nucleus'),('earth orbits the','sun'),('the moon orbits the','earth'),
('the largest planet in the solar system is','jupiter'),('plants use light in','photosynthesis')]
REPHRASE=[
('in mechanics mass times acceleration produces','force'),('force is measured using the unit called','newton'),
('energy uses the si unit called','joule'),('water contains hydrogen plus','oxygen'),('six protons identifies','carbon'),
('eight protons identifies','oxygen'),('earth travels around the','sun'),('the moon travels around the','earth')]

SYL1=['nar','vel','tor','lum','kas','mer','sol','dav','pir','zan','fel','rin','bos','kel','vim','jor','tav','qul','ser','pan','dru','hel','cor','nis']
SYL2=['a','en','or','is','um','et','on','ar','il','os','ex','un']

def unique_words(n,seed):
    rng=random.Random(seed); pool=[a+b for a in SYL1 for b in SYL2]; rng.shuffle(pool); return pool[:n]

def make_chains(seed=0,n_demo=20,n_probe=10):
    names=unique_words((n_demo+n_probe)*3,seed+818); demo=[]; probe=[]; rows=[]; k=0
    for i in range(n_demo+n_probe):
        x,y,z=names[k:k+3]; k+=3
        premises=[f'{x} contains {y}.',f'{y} maps to {z}.']
        if i<n_demo:
            # Preserve v4.2 curriculum exactly: sentences are independent training items.
            demo += (premises+[f'{x} ultimately maps to {z}.',f'{y} is contained by {x}.'])*3
            rows.append(('demo',x,y,z))
        else:
            # Probe target and inverse sentences are deliberately absent.
            probe += premises*4
            rows.append(('probe',x,y,z))
    return demo,probe,pd.DataFrame(rows,columns=['kind','x','y','z'])

def make_training_corpus(seed=0,contextual_demo=False):
    rng=random.Random(seed); s=[]
    for p,a in FACTS: s += [f'{p} {a}.',f'fact: {p} {a}.',f'remember {p} {a}.']
    demo,probe,chains=make_chains(seed); s += demo+probe
    if contextual_demo:
        # Schema-ingestion patch: demonstration entities additionally appear in a contiguous
        # premise->premise->composition context. Probe entities still NEVER receive the
        # composed target or a combined two-premise paragraph during training.
        for _,r in chains[chains.kind=='demo'].iterrows():
            para=f'{r.x} contains {r.y}. {r.y} maps to {r.z}. {r.x} ultimately maps to {r.z}.'
            s += [para,para]
    for i in range(80): s.append(f'noise{i} drift{i} hollow{i} amber{i} sevenfold{i}.')
    rng.shuffle(s)
    return s,chains

def proxy_stream(seed=0,n=180):
    rng=random.Random(seed+500)
    names=['arden','cassian','lyra','orion','mara','selene','tavian','viola']
    verbs=['whispers','wanders','argues','laughs','waits','returns','dreams','speaks']
    objs=['beside the gate','through the hall','near the river','under the tower','after the feast','beyond the garden','before the crowd','under the moon']
    return [f'{rng.choice(names)} {rng.choice(verbs)} {rng.choice(objs)}.' for _ in range(n)]

def download_natural():
    urls={'shakespeare':'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt',
          'bible':'https://www.gutenberg.org/cache/epub/10900/pg10900.txt'}
    out={}
    for name,url in urls.items():
        p=ROOT/f'{name}_corpus.txt'
        if not p.exists():
            print('downloading',name); urllib.request.urlretrieve(url,p)
        text=p.read_text(errors='ignore')
        out[name]=[q.strip().replace('\n',' ') for q in re.split(r'(?<=[.!?])\s+|\n{2,}',text) if 4<=len(tokenize(q))<=70]
    return out

def natural_stream(seed=0,corpus='interleaved',per_source=700):
    d=download_natural(); rng=random.Random(seed+900)
    A=rng.sample(d['shakespeare'],min(per_source,len(d['shakespeare'])))
    B=rng.sample(d['bible'],min(per_source,len(d['bible'])))
    if corpus=='shakespeare': return A
    if corpus=='bible': return B
    out=[]; n=min(len(A),len(B))
    for a,b in zip(A[:n],B[:n]): out += ([a,b] if rng.random()<.5 else [b,a])
    return out

class Vocab:
    def __init__(self,texts,max_vocab=10000):
        c=Counter()
        for t in texts:c.update(tokenize(t))
        self.itos=['<pad>','<unk>']+[w for w,_ in c.most_common(max_vocab-2)]
        self.stoi={w:i for i,w in enumerate(self.itos)}
    def encode(self,s):return [self.stoi.get(w,1) for w in tokenize(s)]

class TinySSM(nn.Module):
    def __init__(self,V,emb=36,hid=56):
        super().__init__();self.emb=nn.Embedding(V,emb);self.rnn=nn.RNN(emb,hid,nonlinearity='tanh',batch_first=True);self.out=nn.Linear(hid,V)
    def forward(self,x,h=None):z,h=self.rnn(self.emb(x),h);return self.out(z),h

def dev(model):return next(model.parameters()).device

def train_sentence(model,opt,ids):
    if len(ids)<2:return np.nan
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    opt.zero_grad(set_to_none=True);logits,_=model(x);loss=nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1));loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),1.0);opt.step();return float(loss.item())

def seq_loss(model,ids):
    if len(ids)<2:return 0.
    d=dev(model);x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:];y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return float(nn.functional.cross_entropy(logits.reshape(-1,logits.size(-1)),y.reshape(-1)).item())

def continuation_loss(model,vocab,prefix,target):
    """Cross-entropy on target tokens only, after conditioning on prefix."""
    pids=vocab.encode(prefix);tids=vocab.encode(target)
    if not pids or not tids:return 0.0
    ids=pids+tids;d=dev(model)
    x=torch.tensor(ids[:-1],dtype=torch.long,device=d)[None,:]
    y=torch.tensor(ids[1:],dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    start=max(0,len(pids)-1);end=min(logits.shape[1],start+len(tids))
    if end<=start:return 0.0
    return float(nn.functional.cross_entropy(logits[:,start:end,:].reshape(-1,logits.size(-1)),y[:,start:end].reshape(-1)).item())

def logits_for_prompt(model,vocab,prompt):
    ids=vocab.encode(prompt);d=dev(model);x=torch.tensor(ids,dtype=torch.long,device=d)[None,:]
    with torch.no_grad():logits,_=model(x)
    return logits[0,-1]

def token_rank(model,vocab,prompt,answer):
    v=logits_for_prompt(model,vocab,prompt);aid=vocab.stoi.get(answer,1);return int((v>v[aid]).sum().item())+1

def top_tokens(model,vocab,prompt,k=5):
    v=logits_for_prompt(model,vocab,prompt);return [vocab.itos[i] for i in torch.topk(v,k=min(k,len(vocab.itos))).indices.tolist()]

def candidate_rank(model,vocab,prompt,true_answer,candidates):
    v=logits_for_prompt(model,vocab,prompt)
    scores=[]
    for a in candidates:
        scores.append((a,float(v[vocab.stoi.get(a,1)].item())))
    scores.sort(key=lambda x:x[1],reverse=True)
    order=[a for a,_ in scores]
    return order.index(true_answer)+1,dict(scores)

# -----------------------------------------------------------------------------
# Candidate evidence and v3.21-style dependency/factor inference
# -----------------------------------------------------------------------------

def unique_candidate_table(train,vocab,early_model,mid_model,final_model):
    """Build candidate evidence from observed consequences, without semantic labels."""
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());N=len(train)
    df=Counter()
    for s in docs:df.update(set(content_tokens(s)))
    token_sets=[set(content_tokens(s)) for s in train]
    rows=[]
    for idx,s in enumerate(docs):
        toks=content_tokens(s);st=set(toks);pp=pos[s];first=min(pp);last=max(pp)
        recurrence=math.log1p(len(pp))
        token_persistence=float(np.mean([math.log1p(df[t]) for t in toks])) if toks else 0.
        cross_context=float(np.mean([df[t]>=3 for t in toks])) if toks else 0.
        temporal_span=(last-first)/max(1,N-1) if len(pp)>1 else 0.
        early_pred=-seq_loss(early_model,vocab.encode(s));mid_pred=-seq_loss(mid_model,vocab.encode(s));final_pred=-seq_loss(final_model,vocab.encode(s))
        gain_early_mid=mid_pred-early_pred;gain_mid_final=final_pred-mid_pred

        future_idx=list(range(first+1,N));supports=[]
        if st and future_idx:
            for w in np.array_split(np.asarray(future_idx,dtype=int),3):
                seen=set()
                for j in w.tolist():seen |= token_sets[j]
                supports.append(len(st&seen)/max(1,len(st)))
        future_support=float(np.mean(supports)) if supports else 0.

        maxjac=0.;seen_prev=set()
        if st:
            for j in range(first):
                if train[j] in seen_prev:continue
                seen_prev.add(train[j]);ps=token_sets[j]
                if ps:maxjac=max(maxjac,len(st&ps)/(len(st|ps)+1e-12))
        novelty=1.-maxjac

        rel=[];unrel=[]
        for j in range(first+1,N):
            if train[j]==s:continue
            ov=len(st&token_sets[j]) if st else 0
            if ov>0:rel.append((ov/(len(st|token_sets[j])+1e-12),j))
            elif len(unrel)<6:unrel.append(j)
        rg=[]
        for _,j in sorted(rel,reverse=True)[:2]:
            t=train[j];rg.append(seq_loss(final_model,vocab.encode(t))-continuation_loss(final_model,vocab,s,t))
        context_gain=float(np.mean(rg)) if rg else 0.
        ug=[]
        for j in unrel[:2]:
            t=train[j];ug.append(continuation_loss(final_model,vocab,s,t)-seq_loss(final_model,vocab.encode(t)))
        low_interference=-float(np.mean(ug)) if ug else 0.

        rows.append({'uid':idx,'text':s,'first_pos':first,'last_pos':last,'count':len(pp),
                     'recurrence':recurrence,'token_persistence':token_persistence,'cross_context':cross_context,
                     'temporal_span':temporal_span,'early_predictability':early_pred,'mid_predictability':mid_pred,'future_predictability':final_pred,
                     'gain_early_mid':gain_early_mid,'gain_mid_final':gain_mid_final,'future_support':future_support,
                     'context_gain':context_gain,'novelty':novelty,'low_interference':low_interference})
    return pd.DataFrame(rows)

SOURCE_COLS=['recurrence','token_persistence','cross_context','temporal_span','early_predictability']
CONSEQUENCE_COLS=['gain_early_mid','gain_mid_final','future_support','context_gain']

def zcols(X):
    X=np.asarray(X,float);mu=X.mean(0);sd=X.std(0);active=sd>1e-8;Z=np.zeros_like(X)
    Z[:,active]=(X[:,active]-mu[active])/(sd[active]+1e-9)
    return Z,mu,sd,active

def connected_components_from_corr(C,threshold=.72):
    n=C.shape[0];adj={i:set() for i in range(n)}
    for i in range(n):
        for j in range(i+1,n):
            if C[i,j]>threshold:adj[i].add(j);adj[j].add(i)
    comps=[];seen=set()
    for i in range(n):
        if i in seen:continue
        st=[i];c=set()
        while st:
            x=st.pop()
            if x in seen:continue
            seen.add(x);c.add(x);st.extend(adj[x]-seen)
        if len(c)>=2:comps.append(c)
    return comps

def learn_multifactor(cov,maxf=3):
    n=cov.shape[0];vals,vecs=np.linalg.eigh(cov);o=np.argsort(vals)[::-1];vals=vals[o];vecs=vecs[:,o]
    base=np.median(vals[min(maxf,n):])+1e-9 if n>maxf else 1e-9
    groups=[];loads=[]
    for k in range(min(maxf,n)):
        strength=max(0.,float((vals[k]-base)/(vals[k]+1e-9)))
        v=np.abs(vecs[:,k]);mx=v.max()+1e-12;g={i for i,l in enumerate(v) if l/mx>.58}
        if len(g)>=2 and strength>.22 and not any(len(g&h)/len(g|h)>.75 for h in groups):
            groups.append(g);loads.append((v,strength))
    return groups,loads,vals

def cap_weights(raw,g):
    g=[i for i in g if raw[i]>0]
    if len(g)<=1:return raw
    total=sum(raw[i] for i in g);cap=max(raw[i] for i in g);f=cap/(total+1e-12)
    for i in g:raw[i]*=f
    return raw

def evidence_model(cand,mode='factor'):
    X=cand[SOURCE_COLS].to_numpy(float);Z,mu,sd,active=zcols(X)
    y=cand['future_predictability'].to_numpy(float);yz=(y-y.mean())/(y.std()+1e-9)
    residual=Z-yz[:,None]
    bias=residual.mean(0)
    rho=np.mean(np.exp(-np.abs(residual)/.8),axis=0)
    cov=np.cov((residual-residual.mean(0)).T)+np.eye(len(SOURCE_COLS))*1e-6
    rsd=np.sqrt(np.maximum(np.diag(cov),1e-9));corr=np.clip(cov/(rsd[:,None]*rsd[None,:]+1e-12),-1,1)
    comps=connected_components_from_corr(corr,.72)
    groups,loads,eigs=learn_multifactor(cov,3)
    raw={i:float(rho[i]**3) if active[i] else 0. for i in range(len(SOURCE_COLS))}
    if mode in ('graph','factor'):
        for g in comps:raw=cap_weights(raw,g)
    if mode=='factor':
        for g,(load,strength) in zip(groups,loads):
            raw=cap_weights(raw,g);mx=max(load[i] for i in g)+1e-12
            for i in g:raw[i]/=(1+1.15*strength*load[i]/mx)
    w=np.array([raw[i] for i in range(len(SOURCE_COLS))]);w/=w.sum()+1e-12
    deb=Z-bias[None,:];U=deb@w
    disagreement=np.sum(w[None,:]*(deb-U[:,None])**2,axis=1)
    # v2-style common-mode risk: don't 'correct' value; lower confidence when many channels miss together.
    substantial=np.abs(residual)>.8
    same_sign=np.maximum((residual>.8).sum(1),(residual<-.8).sum(1))
    h=float(np.mean(same_sign>=max(2,int(math.ceil(len(SOURCE_COLS)/2)))))
    conf=np.exp(-disagreement)*(1-min(.8,h))
    U01=1/(1+np.exp(-np.clip(U,-12,12)))
    out=cand.copy();out['utility_z']=U;out['utility01']=U01;out['evidence_confidence']=conf;out['future_yz']=yz
    meta={'source_cols':SOURCE_COLS,'rho':rho.tolist(),'bias':bias.tolist(),'weights':w.tolist(),'corr':corr.tolist(),
          'graph_components':[[SOURCE_COLS[i] for i in sorted(g)] for g in comps],
          'factor_groups':[[SOURCE_COLS[i] for i in sorted(g)] for g in groups],
          'factor_strengths':[float(x[1]) for x in loads],'factor_loadings':[x[0].tolist() for x in loads],
          'common_cause_risk':h,'eigenvalues':eigs.tolist()}
    return out,meta

def rank_centered(x):
    x=np.asarray(x,float);r=pd.Series(x).rank(method='average',pct=True).to_numpy(float)
    return 2.*(r-.5)

def multi_consequence_model(cand,mode='factor'):
    """Dependency-discount multiple directly observed consequence channels."""
    Y=np.column_stack([rank_centered(cand[c].to_numpy(float)) for c in CONSEQUENCE_COLS])
    active=np.std(Y,axis=0)>1e-8
    cov=np.cov(Y.T)+np.eye(len(CONSEQUENCE_COLS))*1e-6
    sd=np.sqrt(np.maximum(np.diag(cov),1e-9));corr=np.clip(cov/(sd[:,None]*sd[None,:]+1e-12),-1,1)
    comps=connected_components_from_corr(corr,.72);groups,loads,eigs=learn_multifactor(cov,3)
    raw={i:(1. if active[i] else 0.) for i in range(len(CONSEQUENCE_COLS))}
    if mode in ('graph','factor'):
        for g in comps:raw=cap_weights(raw,g)
    if mode=='factor':
        for g,(load,strength) in zip(groups,loads):
            raw=cap_weights(raw,g);mx=max(load[i] for i in g)+1e-12
            for i in g:raw[i]/=(1+1.15*strength*load[i]/mx)
    w=np.array([raw[i] for i in range(len(CONSEQUENCE_COLS))],float);w/=w.sum()+1e-12
    U=Y@w;disagreement=np.sum(w[None,:]*(Y-U[:,None])**2,axis=1)
    same_bad=(Y<-.40).sum(1);h=float(np.mean(same_bad>=max(2,int(math.ceil(len(CONSEQUENCE_COLS)/2)))))
    conf=np.exp(-disagreement)*(1-min(.8,h));U01=1/(1+np.exp(-2*np.clip(U,-6,6)))
    out=cand.copy();out['utility_z']=U;out['utility01']=U01;out['evidence_confidence']=conf
    # Keep novelty and interference structurally distinct from consequence value.
    # Novelty can soften a replacement wall, but it cannot make a bad candidate good.
    # Interference is a cost/constraint, not a positive evidence vote.
    out['novelty01']=(rank_centered(out['novelty'].to_numpy(float))+1.)/2.
    out['interference_risk01']=(rank_centered(-out['low_interference'].to_numpy(float))+1.)/2.
    meta={'consequence_cols':CONSEQUENCE_COLS,'weights':w.tolist(),'corr':corr.tolist(),
          'graph_components':[[CONSEQUENCE_COLS[i] for i in sorted(g)] for g in comps],
          'factor_groups':[[CONSEQUENCE_COLS[i] for i in sorted(g)] for g in groups],
          'factor_strengths':[float(x[1]) for x in loads],'factor_loadings':[x[0].tolist() for x in loads],
          'common_cause_risk':h,'eigenvalues':eigs.tolist()}
    return out,meta

def portfolio_model(cand_v43,cand_multi):
    """Keep alternative consequence models as parallel support lanes.

    A candidate may earn permanence from either lane; one scalar model is not
    allowed to erase a structurally different useful profile. No semantic labels
    define the lanes: one is the prior predictive structural estimator, the other
    is direct multi-consequence evidence.
    """
    a=cand_v43.sort_values('uid').reset_index(drop=True);b=cand_multi.sort_values('uid').reset_index(drop=True)
    assert np.array_equal(a.uid.to_numpy(),b.uid.to_numpy())
    out=b.copy();ua=a.utility01.to_numpy(float);ub=b.utility01.to_numpy(float);qa=a.evidence_confidence.to_numpy(float);qb=b.evidence_confidence.to_numpy(float)
    choose_b=ub>=ua
    out['utility_v43']=ua;out['utility_multi']=ub;out['confidence_v43']=qa;out['confidence_multi']=qb
    out['utility01']=np.maximum(ua,ub);out['evidence_confidence']=np.where(choose_b,qb,qa);out['portfolio_lane']=np.where(choose_b,'multi','v43')
    return out

# -----------------------------------------------------------------------------
# Parity-protected raw templates + comparative replacement bank
# Address != template: address is transparent IDF token signature; template is raw ids.
# -----------------------------------------------------------------------------

class Slot:
    PRIME=1000003
    def __init__(self,text,ids,uid,U,S,Q,count,first_pos,created_step):
        self.text=text;self.a=list(ids);self.b=list(ids);self.uid=uid;self.U=float(U);self.S=float(S);self.Q=float(Q);self.count=int(count);self.first_pos=int(first_pos);self.created_step=int(created_step)
        self.p,self.q=self.chk(self.a);self.corrupted=False
    @classmethod
    def chk(cls,ids):
        q=np.asarray(ids,dtype=np.int64);return int(q.sum()%cls.PRIME),int(((np.arange(len(q))+1)*q).sum()%cls.PRIME)
    def corrupt(self,rng):
        if self.a:
            j=rng.randrange(len(self.a));self.a[j]=(self.a[j]+rng.randint(1,19))%100000;self.corrupted=True
    def retrieve(self,parity=True):
        if self.chk(self.a)==(self.p,self.q):return list(self.a),'primary'
        if parity and self.chk(self.b)==(self.p,self.q):return list(self.b),'repaired'
        return None,'lost'

class SemanticBank:
    def __init__(self,capacity,vocab,idf,comparative=True,mode='factor'):
        self.capacity=capacity;self.vocab=vocab;self.idf=idf;self.comparative=comparative;self.mode=mode;self.slots=[];self.writes=0;self.replacements=0;self.blocked=0;self.holds=0;self.margin=0.
    def retained_value(self,slot,step):
        age=max(0,step-slot.created_step)/max(1,step+1)
        return slot.U*math.sqrt(max(1e-9,math.log1p(slot.count)))/(1+.15*age)
    def candidate_value(self,row):
        return float(row.utility01*(.65+.35*row.stability)*(.75+.25*row.evidence_confidence))
    def admit(self,row,step,margin):
        if row.evidence_confidence <= 0 or row.utility01<.5:
            self.holds+=1;return False
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=[self.retained_value(s,step) for s in self.slots];j=int(np.argmin(vals))
        if self.comparative:
            if c-vals[j] <= margin:self.blocked+=1;return False
        else:
            # Old/automatic full-bank behavior: qualifying candidate evicts weakest.
            if c<.5:self.blocked+=1;return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;return True
    def corrupt(self,p,seed):
        rng=random.Random(seed)
        for s in self.slots:
            if rng.random()<p:s.corrupt(rng)
    def replay_ids(self,parity=True):
        out=[];repair=lost=0
        for s in self.slots:
            ids,status=s.retrieve(parity)
            if ids is not None:out.append(ids)
            repair+=status=='repaired';lost+=status=='lost'
        return out,repair,lost
    def query_score(self,q_tokens,slot):
        q=set(q_tokens);s=set(content_tokens(slot.text));ov=q&s
        if not ov:return 0.
        num=sum(self.idf.get(t,1.) for t in ov);den=sum(self.idf.get(t,1.) for t in q)+1e-12
        return float(num/den)
    def retrieve_templates(self,prompt,max_hops=2,parity=True):
        # Honest associative retrieval: never force a guess on tied/no-overlap addresses.
        q=content_tokens(prompt);chosen=[];seen=set();logs=[]
        for hop in range(max_hops):
            scores=[]
            for i,s in enumerate(self.slots):
                if i in seen:continue
                scores.append((self.query_score(q,s),i))
            if not scores:break
            scores.sort(reverse=True);best,bi=scores[0];second=scores[1][0] if len(scores)>1 else 0.
            # Dynamic margin: rare-token exact overlap should separate from the next address.
            margin=max(.025,.10*best)
            if best<=0 or best-second<=margin:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin});break
            slot=self.slots[bi];ids,status=slot.retrieve(parity)
            if ids is None:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin,'status':'lost'});break
            chosen.append(slot.text);seen.add(bi);q += content_tokens(slot.text)
            logs.append({'hop':hop,'resolved':True,'best':best,'second':second,'margin':margin,'status':status,'text':slot.text,'uid':slot.uid})
        return chosen,logs


class AdaptiveSemanticBank(SemanticBank):
    """Bounded soft-wall replacement: churn stiffens; persistent blocking softens."""
    def __init__(self,capacity,vocab,idf,comparative=True,adaptive=True,mode='v44'):
        super().__init__(capacity,vocab,idf,comparative,mode)
        self.adaptive=adaptive;self.base_margin=0.;self.current_margin=0.;self.margin_min=np.inf;self.margin_max=0.
        self.replace_ema=0.;self.block_ema=0.;self.pressure_alpha=.08
    def candidate_value(self,row):
        return float(row.utility01*(.65+.35*row.evidence_confidence)*(.75+.25*(1.-float(getattr(row,'interference_risk01',0.)))))
    def retained_value(self,slot,step):
        age=max(0,step-slot.created_step)/max(1,step+1)
        return float(slot.U*(.90+.10*slot.Q)/(1+.08*age))
    def _pressure(self,event):
        a=self.pressure_alpha
        self.replace_ema=(1-a)*self.replace_ema+a*(1. if event=='replace' else 0.)
        self.block_ema=(1-a)*self.block_ema+a*(1. if event=='block' else 0.)
        if self.adaptive:
            scale=float(np.clip(math.exp(1.25*(self.replace_ema-self.block_ema)),.25,2.5))
            self.current_margin=self.base_margin*scale
        self.margin_min=min(self.margin_min,self.current_margin);self.margin_max=max(self.margin_max,self.current_margin)
    def admit(self,row,step,margin=None):
        if row.evidence_confidence<=0 or row.utility01<.5:
            self.holds+=1;return False
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=[self.retained_value(s,step) for s in self.slots];j=int(np.argmin(vals));m=self.current_margin if self.comparative else 0.
        # Novelty is pressure for admission, not utility. It may reduce the extra
        # replacement margin, but never allows a lower-value candidate to win.
        novelty=float(getattr(row,'novelty01',0.));m_eff=m*(1.-.45*novelty)
        if self.comparative and c-vals[j]<=m_eff:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

def multi_gate_margin(cand):
    C=cand.utility01.to_numpy(float)*(.65+.35*cand.evidence_confidence.to_numpy(float))*(.75+.25*(1.-cand.interference_risk01.to_numpy(float)))
    q25,q75=np.quantile(C,[.25,.75]);return float(.18*max(1e-6,q75-q25))

def build_v44_bank(cand,vocab,kind='multi_adaptive_gate',capacity=96):
    idf=compute_idf(cand);comparative=('auto' not in kind);adaptive=('adaptive' in kind)
    bank=AdaptiveSemanticBank(capacity,vocab,idf,comparative=comparative,adaptive=adaptive,mode=kind)
    base=multi_gate_margin(cand);bank.base_margin=base;bank.current_margin=base;bank.margin=base;bank.margin_min=base;bank.margin_max=base
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,base)
    bank.margin=bank.current_margin
    return bank

class DualLaneBank(AdaptiveSemanticBank):
    """Shared-capacity bank that preserves emergent evidence lanes without semantic quotas."""
    def __init__(self,capacity,vocab,idf,mode='dual_lane_adaptive'):
        super().__init__(capacity,vocab,idf,comparative=True,adaptive=True,mode=mode)
        self.lane_pressure=Counter()
    def _lane(self,row):
        return str(getattr(row,'portfolio_lane','single'))
    def admit(self,row,step,margin=None):
        if row.evidence_confidence<=0 or row.utility01<.5:
            self.holds+=1;return False
        lane=self._lane(row);self.lane_pressure[lane]+=1
        c=self.candidate_value(row)
        slot=Slot(row.text,self.vocab.encode(row.text),int(row.uid),float(row.utility01),float(row.stability),float(row.evidence_confidence),int(row['count']),int(row.first_pos),step);slot.lane=lane
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        lanes=set(self.lane_pressure)|{getattr(x,'lane','single') for x in self.slots}
        total=max(1,sum(self.lane_pressure.values()))
        desired={l:max(1,int(round(self.capacity*self.lane_pressure[l]/total))) for l in lanes}
        # Repair rounding to capacity deterministically.
        while sum(desired.values())>self.capacity:
            l=max(sorted(desired),key=lambda x:desired[x]);desired[l]-=1
        while sum(desired.values())<self.capacity:
            l=max(sorted(desired),key=lambda x:self.lane_pressure[x]);desired[l]+=1
        counts=Counter(getattr(x,'lane','single') for x in self.slots)
        if counts[lane] < desired.get(lane,0):
            over=[l for l in lanes if counts[l]>desired.get(l,0)]
            pool=[i for i,x in enumerate(self.slots) if getattr(x,'lane','single') in over] if over else list(range(len(self.slots)))
        else:
            pool=[i for i,x in enumerate(self.slots) if getattr(x,'lane','single')==lane]
            if not pool:pool=list(range(len(self.slots)))
        vals=[self.retained_value(self.slots[i],step) for i in pool];pi=int(np.argmin(vals));j=pool[pi];weak=vals[pi]
        m=self.current_margin;novelty=float(getattr(row,'novelty01',0.));m_eff=m*(1.-.45*novelty)
        if c-weak<=m_eff:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

def build_dual_lane_bank(cand,vocab,capacity=96):
    idf=compute_idf(cand);bank=DualLaneBank(capacity,vocab,idf)
    base=multi_gate_margin(cand);bank.base_margin=base;bank.current_margin=base;bank.margin=base;bank.margin_min=base;bank.margin_max=base
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,base)
    bank.margin=bank.current_margin
    return bank

class SplitRoleBank:
    """One total memory budget, two learned operational roles.

    Replay/consolidation and query-time addressing compete under separate half-budget
    banks because the local ablation showed they optimize different consequences.
    This 50/50 split is an explicit scaffold and is logged for later adaptive removal.
    """
    def __init__(self,replay_bank,query_bank,total_capacity):
        self.replay_bank=replay_bank;self.query_bank=query_bank;self.capacity=total_capacity;self.mode='split_role'
    @property
    def slots(self):return self.query_bank.slots
    @property
    def writes(self):return self.replay_bank.writes+self.query_bank.writes
    @property
    def replacements(self):return self.replay_bank.replacements+self.query_bank.replacements
    @property
    def blocked(self):return self.replay_bank.blocked+self.query_bank.blocked
    @property
    def holds(self):return self.replay_bank.holds+self.query_bank.holds
    @property
    def margin(self):return (getattr(self.replay_bank,'margin',0.)+getattr(self.query_bank,'margin',0.))/2
    @property
    def margin_min(self):return min(getattr(self.replay_bank,'margin_min',self.margin),getattr(self.query_bank,'margin_min',self.margin))
    @property
    def margin_max(self):return max(getattr(self.replay_bank,'margin_max',self.margin),getattr(self.query_bank,'margin_max',self.margin))
    @property
    def replace_ema(self):return getattr(self.replay_bank,'replace_ema',0.)+getattr(self.query_bank,'replace_ema',0.)
    @property
    def block_ema(self):return getattr(self.replay_bank,'block_ema',0.)+getattr(self.query_bank,'block_ema',0.)
    def corrupt(self,p,seed):
        self.replay_bank.corrupt(p,seed);self.query_bank.corrupt(p,seed+100003)
    def replay_ids(self,parity=True):
        a,ra,la=self.replay_bank.replay_ids(parity);_,rb,lb=self.query_bank.replay_ids(parity);return a,ra+rb,la+lb
    def retrieve_templates(self,prompt,max_hops=2,parity=True):
        return self.query_bank.retrieve_templates(prompt,max_hops,parity)

def build_split_role_bank(cand_v43,cand_multi,vocab,capacity=96):
    replay_cap=capacity//2;query_cap=capacity-replay_cap
    replay=build_v44_bank(cand_multi,vocab,'multi_auto',replay_cap)
    query=build_bank(cand_v43,{},vocab,'factor_gate',query_cap)
    return SplitRoleBank(replay,query,capacity)

def compute_idf(cand):
    docs=[set(content_tokens(s)) for s in cand.text];df=Counter();
    for d in docs:df.update(d)
    N=len(docs);return {t:(math.log((N+1)/(c+1))+1)**2 for t,c in df.items()}

def add_stability(cand):
    x=cand.copy();span=x.temporal_span.to_numpy(float);cnt=np.log1p(x['count'].to_numpy(float));cnt=cnt/(cnt.max()+1e-12);x['stability']=np.clip(.5*span+.5*cnt,0,1);return x

def gate_margin(cand):
    # Field report says the margin is calibration-dependent, not universal.
    # Here it scales to candidate-value dispersion using no semantic labels.
    C=cand.utility01.to_numpy(float)*(.65+.35*cand.stability.to_numpy(float))*(.75+.25*cand.evidence_confidence.to_numpy(float))
    return float(.25*np.std(C))

def build_bank(cand,meta,vocab,kind='factor_gate',capacity=96):
    idf=compute_idf(cand);bank=SemanticBank(capacity,vocab,idf,comparative=('auto' not in kind),mode=kind)
    margin=gate_margin(cand);bank.margin=margin
    # Process in observational order, not best-first.
    for step,row in cand.sort_values('first_pos').iterrows():bank.admit(row,step,margin)
    return bank

def old_v42_bank(train,vocab,final_model,capacity=96):
    # v4.2 selector, but bank is unique-template bounded for fair address retrieval.
    pos=defaultdict(list)
    for i,s in enumerate(train):pos[s].append(i)
    docs=list(pos.keys());counts=Counter();toks=[]
    for s in docs:
        q=content_tokens(s);toks.append(q);counts.update(set(q))
    rows=[]
    for i,(s,q) in enumerate(zip(docs,toks)):
        rows.append({'uid':i,'text':s,'first_pos':min(pos[s]),'count':len(pos[s]),
                     'recurrence':np.mean([math.log1p(counts[w]) for w in q]) if q else 0.,
                     'cross_context':np.mean([counts[w]>=3 for w in q]) if q else 0.,
                     'predictability':-seq_loss(final_model,vocab.encode(s))})
    d=pd.DataFrame(rows);X=d[['recurrence','cross_context','predictability']].to_numpy(float);Z,_,_,active=zcols(X)
    C=np.zeros((3,3));idx=np.where(active)[0]
    if len(idx)>1:
        Q=np.nan_to_num(np.abs(np.corrcoef(Z[:,idx],rowvar=False)));np.fill_diagonal(Q,0)
        for a,i in enumerate(idx):
            for b,j in enumerate(idx):C[i,j]=Q[a,b]
    rho=np.where(active,.7,0);a=rho**3;base=a/(a.sum()+1e-12);w=a/(1+1.5*(C@base));w/=w.sum()+1e-12;score=Z@w
    sets=[set(content_tokens(s)) for s in docs];selected=[];remain=set(range(len(d)))
    while remain and len(selected)<min(capacity,len(d)):
        best=None;bv=-1e99
        for i in remain:
            red=max((len(sets[i]&sets[j])/(len(sets[i]|sets[j])+1e-12) for j in selected),default=0.);v=float(score[i])-.85*red
            if v>bv:bv=v;best=i
        selected.append(best);remain.remove(best)
    # Reuse bank container, direct add selected templates.
    idf=compute_idf(pd.DataFrame({'text':docs}));bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='v42_selector')
    for step,i in enumerate(selected):
        row=d.iloc[i];slot=Slot(row.text,vocab.encode(row.text),int(row.uid),.75,.75,.75,int(row['count']),int(row.first_pos),step);bank.slots.append(slot);bank.writes+=1
    return bank

def random_bank(train,vocab,capacity,seed):
    docs=list(dict.fromkeys(train));rng=random.Random(seed);picks=rng.sample(docs,min(capacity,len(docs)));tmp=pd.DataFrame({'text':docs});idf=compute_idf(tmp);bank=SemanticBank(capacity,vocab,idf,comparative=False,mode='random')
    pos={s:i for i,s in enumerate(docs)}
    for step,s in enumerate(picks):bank.slots.append(Slot(s,vocab.encode(s),pos[s],.5,.5,.5,1,pos[s],step));bank.writes+=1
    return bank

# -----------------------------------------------------------------------------
# Binding-specific probes. No chain ids/answers enter training or memory selection.
# -----------------------------------------------------------------------------

def augment_prompt(bank,prompt,parity=True,order='path'):
    if bank is None:return prompt,[],[]
    templates,logs=bank.retrieve_templates(prompt,2,parity)
    if order=='reverse':templates=list(reversed(templates))
    aug=(' '.join(templates)+' '+prompt).strip() if templates else prompt
    return aug,templates,logs

def probe_panel(model,vocab,chains,seed,stage,variant,bank=None,parity=True):
    rows=[]
    def add(kind,prompt,ans,item,mode='plain',candidates=None,wrong=None):
        aug=prompt;retr=[];logs=[]
        if mode in ('assist','assist_reverse') and bank is not None:
            aug,retr,logs=augment_prompt(bank,prompt,parity,'reverse' if mode=='assist_reverse' else 'path')
        elif mode in ('first_only','second_only') and bank is not None:
            # Single-premise controls separate retrieval from composition.
            _,rr,ll=augment_prompt(bank,prompt,parity,'path');retr=(rr[:1] if mode=='first_only' else rr[-1:]) if rr else [];logs=ll;aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        r=token_rank(model,vocab,aug,ans);cr=np.nan;margin=np.nan;wrong_score=np.nan;true_score=np.nan
        if candidates:
            cr,sc=candidate_rank(model,vocab,aug,ans,candidates);true_score=sc.get(ans,np.nan)
            if wrong is not None:wrong_score=sc.get(wrong,np.nan);margin=true_score-wrong_score
        rows.append({'seed':seed,'stage':stage,'variant':variant,'probe':kind,'mode':mode,'item':item,'prompt':prompt,'aug_prompt':aug,'answer':ans,
                     'rank':r,'candidate_rank':cr,'binding_margin':margin,'true_score':true_score,'wrong_score':wrong_score,
                     'top1':int(r==1),'top5':int(r<=5),'candidate_top1':int(cr==1) if not np.isnan(cr) else np.nan,
                     'top_tokens':'|'.join(top_tokens(model,vocab,aug,5)),'retrieved':' || '.join(retr),'retrieval_log':json.dumps(logs)})
    for i,(p,a) in enumerate(FACTS):add('exact_fact',p,a,f'fact{i}')
    for i,(p,a) in enumerate(REPHRASE):add('rephrase',p,a,f'rephrase{i}')
    q=chains[chains.kind=='probe'].reset_index(drop=True);zcands=q.z.tolist();xcands=q.x.tolist()
    for i,row in q.iterrows():
        wrong=q.iloc[(i+1)%len(q)]
        for mode in ['plain','assist','assist_reverse','first_only','second_only']:
            add('two_hop',f'{row.x} ultimately maps to',row.z,f'chain{i}',mode,zcands,wrong.z)
        add('two_hop_shuffled',f'{row.x} ultimately maps to',wrong.z,f'chain{i}','plain',zcands,row.z)
        for mode in ['plain','assist']:
            add('inverse',f'{row.y} is contained by',row.x,f'chain{i}',mode,xcands,wrong.x)
        add('premise_contains',f'{row.x} contains',row.y,f'chain{i}')
        add('premise_maps',f'{row.y} maps to',row.z,f'chain{i}')
    return pd.DataFrame(rows)

def summarize_probes(df):
    def safe_cmrr(x):
        a=np.asarray(x,float);a=a[np.isfinite(a)]
        return float(np.mean(1/a)) if len(a) else np.nan
    return df.groupby(['stage','variant','probe','mode']).agg(top1=('top1','mean'),top5=('top5','mean'),candidate_top1=('candidate_top1','mean'),
        mrr=('rank',lambda x:float(np.mean(1/np.asarray(x,float)))),candidate_mrr=('candidate_rank',safe_cmrr),
        binding_margin=('binding_margin','mean'),median_rank=('rank','median'),n=('rank','size')).reset_index()


# =============================================================================
# Adrianic v4.5 — Multi-access memory objects + explicit identity-binding state
# =============================================================================
OUT = ROOT / 'adrianic_v45_multiaccess_binding_results'
OUT.mkdir(parents=True, exist_ok=True)

class MultiAccessSlot(Slot):
    def __init__(self,text,ids,uid,C,R,QC,QR,count,first_pos,created_step):
        super().__init__(text,ids,uid,U=max(float(C),float(R)),S=.5,Q=max(float(QC),float(QR)),
                         count=count,first_pos=first_pos,created_step=created_step)
        self.C=float(C); self.R=float(R); self.QC=float(QC); self.QR=float(QR)

class MultiAccessBank:
    # Confidence remains logged and can HOLD truly unresolved candidates, but it is
    # not allowed to re-multiply an already evidence-adjusted access value into near-zero.
    # The first v4.5 screen showed that doing so destroyed retrieval coverage.
    def __init__(self,capacity,vocab,idf,adaptive=True):
        self.capacity=capacity;self.vocab=vocab;self.idf=idf;self.adaptive=adaptive
        self.slots=[];self.writes=0;self.replacements=0;self.blocked=0;self.holds=0
        self.base_margin=0.;self.current_margin=0.;self.margin=0.
        self.margin_min=np.inf;self.margin_max=0.;self.replace_ema=0.;self.block_ema=0.;self.alpha=.08

    @staticmethod
    def survival(C,R,QC,QR):
        c=float(C)*(.90+.10*float(QC));r=float(R)*(.90+.10*float(QR))
        return max(c,r)

    def _pressure(self,event):
        a=self.alpha
        self.replace_ema=(1-a)*self.replace_ema+a*float(event=='replace')
        self.block_ema=(1-a)*self.block_ema+a*float(event=='block')
        if self.adaptive:
            scale=float(np.clip(math.exp(.85*(self.replace_ema-self.block_ema)),.35,2.0))
            self.current_margin=self.base_margin*scale
        self.margin_min=min(self.margin_min,self.current_margin)
        self.margin_max=max(self.margin_max,self.current_margin)

    def admit(self,row,step):
        C=float(row.C);R=float(row.R);QC=float(row.QC);QR=float(row.QR)
        if (QC<=0 and QR<=0) or max(C,R)<.50:
            self.holds+=1;return False
        slot=MultiAccessSlot(row.text,self.vocab.encode(row.text),int(row.uid),C,R,QC,QR,
                             int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(slot);self.writes+=1;return True
        vals=np.array([self.survival(s.C,s.R,s.QC,s.QR) for s in self.slots])
        j=int(np.argmin(vals));cand=self.survival(C,R,QC,QR)
        nov=float(getattr(row,'novelty01',0.));m=self.current_margin*(1-.35*nov)
        if cand<=vals[j]+m:
            self.blocked+=1;self._pressure('block');return False
        self.slots[j]=slot;self.writes+=1;self.replacements+=1;self._pressure('replace');return True

    def corrupt(self,p,seed):
        rng=random.Random(seed)
        for s in self.slots:
            if rng.random()<p:s.corrupt(rng)

    def replay_ids(self,parity=True):
        ranked=sorted(self.slots,key=lambda s:(s.C*(.90+.10*s.QC),s.created_step),reverse=True)
        out=[];repair=lost=0
        for s in ranked:
            ids,status=s.retrieve(parity)
            if ids is not None:out.append(ids)
            repair+=status=='repaired';lost+=status=='lost'
        return out,repair,lost

    def _address_score(self,q_tokens,slot):
        q=set(q_tokens);st=set(content_tokens(slot.text));ov=q&st
        if not ov:return 0.
        num=sum(self.idf.get(t,1.) for t in ov);den=sum(self.idf.get(t,1.) for t in q)+1e-12
        return float(num/den)*(.30+.70*slot.R)*(.90+.10*slot.QR)

    def retrieve_templates(self,prompt,max_hops=3,parity=True):
        q=content_tokens(prompt);chosen=[];seen=set();logs=[]
        for hop in range(max_hops):
            scores=[(self._address_score(q,s),i) for i,s in enumerate(self.slots) if i not in seen]
            if not scores:break
            scores.sort(reverse=True);best,bi=scores[0];second=scores[1][0] if len(scores)>1 else 0.
            margin=max(.018,.07*best)
            if best<=0 or best-second<=margin:
                logs.append({'hop':hop,'resolved':False,'best':best,'second':second,'margin':margin});break
            s=self.slots[bi];ids,status=s.retrieve(parity)
            if ids is None:break
            chosen.append(s.text);seen.add(bi);q+=content_tokens(s.text)
            logs.append({'hop':hop,'resolved':True,'uid':s.uid,'R':s.R,'C':s.C,'text':s.text,'status':status})
        return chosen,logs

    def identity_join(self,prompt,parity=True,shuffle=False):
        q=content_tokens(prompt)
        if not q:return {},[],{'resolved':False,'reason':'empty_query'}
        anchors=sorted(set(q),key=lambda t:self.idf.get(t,1.),reverse=True)[:max(1,min(2,len(set(q))))]
        first=[]
        for i,s in enumerate(self.slots):
            toks=content_tokens(s.text);aset=set(toks)&set(anchors)
            if aset:first.append((self._address_score(anchors,s),i,aset,toks))
        first.sort(reverse=True)
        if not first:return {},[],{'resolved':False,'reason':'no_anchor_template'}

        path_scores=defaultdict(float);paths=[]
        for fs,fi,aset,atoks in first[:8]:
            A=self.slots[fi];apos={t:j for j,t in enumerate(atoks)}
            bridges=sorted(
                [t for t in set(atoks) if t not in set(q) and self.idf.get(t,1.)>1.05],
                key=lambda t:self.idf.get(t,1.),reverse=True
            )[:8]
            for bridge in bridges:
                valid_anchor=[a for a in aset if apos.get(a,10**9)<apos.get(bridge,-1)]
                if not valid_anchor:continue
                for bi,B in enumerate(self.slots):
                    if bi==fi:continue
                    btoks=content_tokens(B.text)
                    if bridge not in btoks:continue
                    bpos={t:j for j,t in enumerate(btoks)};brp=bpos[bridge]
                    candidates=[t for t in btoks[brp+1:] if t not in set(q) and t!=bridge and self.idf.get(t,1.)>1.05]
                    if not candidates:continue
                    Baddr=(.30+.70*B.R)*(.90+.10*B.QR)
                    bridge_w=min(3.0,self.idf.get(bridge,1.))
                    anchor_w=max(min(3.0,self.idf.get(a,1.)) for a in valid_anchor)
                    for c in set(candidates):
                        dist=max(1,bpos[c]-brp);sc=fs*Baddr*bridge_w*anchor_w/dist
                        path_scores[c]+=float(sc)
                        paths.append({'A':A.text,'B':B.text,'bridge':bridge,'candidate':c,'score':float(sc)})
        if not path_scores:return {},[],{'resolved':False,'reason':'no_identity_path'}
        if shuffle:
            keys=sorted(path_scores);vals=[path_scores[k] for k in keys]
            rng=random.Random(int(hashlib.sha256(prompt.encode()).hexdigest()[:12],16));rng.shuffle(vals)
            path_scores=defaultdict(float,{k:v for k,v in zip(keys,vals)})
        mx=max(path_scores.values())+1e-12
        norm={k:float(v/mx) for k,v in path_scores.items()}
        return norm,sorted(paths,key=lambda x:x['score'],reverse=True)[:12],{'resolved':True,'anchors':anchors,'n_paths':len(paths)}

def build_multi_access_bank(cand_v43,cand_multi,vocab,capacity=96,adaptive=True):
    a=cand_v43.sort_values('uid').reset_index(drop=True);b=cand_multi.sort_values('uid').reset_index(drop=True)
    assert np.array_equal(a.uid.to_numpy(),b.uid.to_numpy())
    merged=b.copy()
    merged['C']=b.utility01.to_numpy(float);merged['QC']=b.evidence_confidence.to_numpy(float)
    merged['R']=a.utility01.to_numpy(float);merged['QR']=a.evidence_confidence.to_numpy(float)
    idf=compute_idf(merged);bank=MultiAccessBank(capacity,vocab,idf,adaptive)
    sv=np.maximum(merged.C*(.90+.10*merged.QC),merged.R*(.90+.10*merged.QR))
    q25,q75=np.quantile(sv,[.25,.75]);bank.base_margin=.12*max(1e-6,q75-q25)
    bank.current_margin=bank.base_margin;bank.margin=bank.current_margin
    bank.margin_min=bank.current_margin;bank.margin_max=bank.current_margin
    for step,row in merged.sort_values('first_pos').iterrows():bank.admit(row,step)
    bank.margin=bank.current_margin
    return bank,merged


class ParetoMultiAccessBank(MultiAccessBank):
    """Online Pareto-retained multi-access bank.

    No fixed replay/retrieval quota and no scalar collapse of C and R.
    When full, candidate + stored points are ranked by nondominated fronts;
    crowding distance preserves useful extremes and removes redundant interior
    points. This is the standard resource geometry, not a semantic rule.
    """
    @staticmethod
    def _point_obj(obj):
        return np.array([
            obj.C*(.90+.10*obj.QC),
            obj.R*(.90+.10*obj.QR)
        ],float)

    @staticmethod
    def _fronts(points):
        n=len(points);dominated=[[] for _ in range(n)];count=np.zeros(n,int);front=[[]]
        for p in range(n):
            for q in range(n):
                if p==q:continue
                pp=points[p];qq=points[q]
                pdom=np.all(pp>=qq) and np.any(pp>qq)
                qdom=np.all(qq>=pp) and np.any(qq>pp)
                if pdom:dominated[p].append(q)
                elif qdom:count[p]+=1
            if count[p]==0:front[0].append(p)
        k=0
        while k<len(front) and front[k]:
            nxt=[]
            for p in front[k]:
                for q in dominated[p]:
                    count[q]-=1
                    if count[q]==0:nxt.append(q)
            k+=1
            if nxt:front.append(nxt)
        return front

    @staticmethod
    def _crowding(points,idxs):
        if not idxs:return {}
        if len(idxs)<=2:return {i:float('inf') for i in idxs}
        d={i:0. for i in idxs};P=np.asarray([points[i] for i in idxs])
        for dim in range(P.shape[1]):
            order=sorted(idxs,key=lambda i:points[i][dim])
            d[order[0]]=d[order[-1]]=float('inf')
            lo=points[order[0]][dim];hi=points[order[-1]][dim]
            if hi-lo<=1e-12:continue
            for j in range(1,len(order)-1):
                if np.isfinite(d[order[j]]):
                    d[order[j]]+=(points[order[j+1]][dim]-points[order[j-1]][dim])/(hi-lo)
        return d

    def admit(self,row,step):
        C=float(row.C);R=float(row.R);QC=float(row.QC);QR=float(row.QR)
        if (QC<=0 and QR<=0) or max(C,R)<.50:
            self.holds+=1;return False
        cand=MultiAccessSlot(row.text,self.vocab.encode(row.text),int(row.uid),C,R,QC,QR,
                             int(row['count']),int(row.first_pos),step)
        if len(self.slots)<self.capacity:
            self.slots.append(cand);self.writes+=1;return True

        objs=self.slots+[cand];points=[self._point_obj(s) for s in objs]
        fronts=self._fronts(points)
        rank={}
        crowd={}
        for ri,f in enumerate(fronts):
            for i in f:rank[i]=ri
            crowd.update(self._crowding(points,f))
        # Worst = deepest Pareto front; within it, most redundant / least crowded.
        worst=max(range(len(objs)),key=lambda i:(rank.get(i,10**9),-crowd.get(i,0.)))
        if worst==len(objs)-1:
            self.blocked+=1;return False
        self.slots[worst]=cand;self.writes+=1;self.replacements+=1;return True

def build_pareto_multi_access_bank(cand_v43,cand_multi,vocab,capacity=96):
    a=cand_v43.sort_values('uid').reset_index(drop=True);b=cand_multi.sort_values('uid').reset_index(drop=True)
    assert np.array_equal(a.uid.to_numpy(),b.uid.to_numpy())
    merged=b.copy();merged['C']=b.utility01.to_numpy(float);merged['QC']=b.evidence_confidence.to_numpy(float)
    merged['R']=a.utility01.to_numpy(float);merged['QR']=a.evidence_confidence.to_numpy(float)
    bank=ParetoMultiAccessBank(capacity,vocab,compute_idf(merged),adaptive=False)
    for step,row in merged.sort_values('first_pos').iterrows():bank.admit(row,step)
    return bank,merged


class IdentityBindingGraph:
    """Compact structural state derived from repeated ordered token identity.

    This is not a semantic parser. It never names subject/predicate/object roles.
    Repeated rare-token order creates directed edges; exact shared identity permits
    deterministic multi-step traversal.
    """
    def __init__(self,cand_v43,max_edges=512):
        self.idf=compute_idf(cand_v43)
        vals=np.array(list(self.idf.values()),float)
        self.rare_floor=float(np.quantile(vals,.60)) if len(vals) else 1.
        edge_weight=defaultdict(float);edge_support=defaultdict(float);edge_sources=defaultdict(int)
        for _,row in cand_v43.iterrows():
            toks=content_tokens(row.text)
            rare=[t for t in toks if self.idf.get(t,1.)>=self.rare_floor]
            # preserve order and remove repeated token occurrences inside one sentence
            clean=[]
            for t in rare:
                if not clean or clean[-1]!=t:clean.append(t)
            if len(clean)<2:continue
            base=float(row.utility01)*(.90+.10*float(row.evidence_confidence))*math.log1p(max(1,int(row['count'])))
            for i in range(len(clean)):
                for j in range(i+1,len(clean)):
                    a,b=clean[i],clean[j]
                    if a==b:continue
                    edge_weight[(a,b)]+=base
                    edge_support[(a,b)]+=int(row['count'])
                    edge_sources[(a,b)]+=1
        # Persistence gate: a structural edge must have recurring support.
        edges=[(k,w,edge_support[k],edge_sources[k]) for k,w in edge_weight.items() if edge_support[k]>=2]
        edges.sort(key=lambda x:(x[1],x[2]),reverse=True);edges=edges[:max_edges]
        self.edges={(a,b):float(w) for (a,b),w,_,_ in edges}
        self.support={(a,b):float(s) for (a,b),_,s,_ in edges}
        self.out=defaultdict(dict)
        for (a,b),w in self.edges.items():self.out[a][b]=w
        self.max_edges=max_edges

    def _anchor(self,prompt):
        q=content_tokens(prompt)
        if not q:return None
        return max(set(q),key=lambda t:self.idf.get(t,1.))

    def one_step(self,prompt):
        x=self._anchor(prompt)
        if x is None:return {},[]
        raw=dict(self.out.get(x,{}));mx=max(raw.values(),default=0.)+1e-12
        return ({k:float(v/mx) for k,v in raw.items()} if raw else {}),[{'x':x,'z':k,'score':v} for k,v in raw.items()]

    def two_step(self,prompt):
        x=self._anchor(prompt)
        if x is None:return {},[]
        raw=defaultdict(float);paths=[]
        for y,w1 in self.out.get(x,{}).items():
            for z,w2 in self.out.get(y,{}).items():
                if z==x:continue
                # geometric mean prevents one huge edge from overwhelming a weak second edge.
                s=math.sqrt(max(0.,w1*w2))
                raw[z]+=s
                paths.append({'x':x,'bridge':y,'z':z,'score':float(s),'w1':float(w1),'w2':float(w2)})
        mx=max(raw.values(),default=0.)+1e-12
        return ({k:float(v/mx) for k,v in raw.items()} if raw else {}),sorted(paths,key=lambda d:d['score'],reverse=True)[:12]

    def shuffled_two_step_mean(self,prompt,n=32):
        # Strong negative control: preserve first-hop edges and the full collection
        # of second-hop adjacency lists, but break the exact bridge identity by
        # permuting which source-node adjacency list is consulted at hop two.
        x=self._anchor(prompt)
        if x is None or x not in self.out:return {}
        source_nodes=sorted(self.out.keys())
        if len(source_nodes)<2:return {}
        base=int(hashlib.sha256(prompt.encode()).hexdigest()[:12],16)
        acc=defaultdict(float)
        for r in range(n):
            perm=list(source_nodes);random.Random(base+r*7919).shuffle(perm)
            mapping=dict(zip(source_nodes,perm))
            raw=defaultdict(float)
            for y,w1 in self.out.get(x,{}).items():
                yy=mapping.get(y,y)
                for z,w2 in self.out.get(yy,{}).items():
                    if z==x:continue
                    raw[z]+=math.sqrt(max(0.,w1*w2))
            mx=max(raw.values(),default=0.)+1e-12
            for z,v in raw.items():acc[z]+=float(v/mx)/n
        return dict(acc)

class GraphBindingSystem:
    """Same replay bank as v44_multi_auto plus a compact identity graph."""
    def __init__(self,replay_bank,graph):
        self.replay_bank=replay_bank;self.identity_graph=graph
        # bank-like audit surface
        self.capacity=replay_bank.capacity;self.slots=replay_bank.slots
        self.writes=replay_bank.writes;self.replacements=replay_bank.replacements
        self.blocked=replay_bank.blocked;self.holds=replay_bank.holds;self.margin=getattr(replay_bank,'margin',0.)
    def corrupt(self,p,seed):self.replay_bank.corrupt(p,seed)
    def replay_ids(self,parity=True):return self.replay_bank.replay_ids(parity)
    def retrieve_templates(self,prompt,max_hops=3,parity=True):return self.replay_bank.retrieve_templates(prompt,max_hops,parity)
    def identity_join(self,prompt,parity=True,shuffle=False):
        if shuffle:
            sc=self.identity_graph.shuffled_two_step_mean(prompt,32);return sc,[],{'resolved':bool(sc),'control':'32-shuffle-mean'}
        sc,paths=self.identity_graph.two_step(prompt);return sc,paths,{'resolved':bool(sc),'edges':len(self.identity_graph.edges)}

def fused_candidate_rank(model,vocab,prompt,true_answer,candidates,struct_scores=None,struct_gain=1.35):
    v=logits_for_prompt(model,vocab,prompt)
    base=np.array([float(v[vocab.stoi.get(a,1)].item()) for a in candidates],float)
    sd=float(np.std(base)+1e-6)
    ss=np.array([0. if struct_scores is None else struct_scores.get(a,0.) for a in candidates],float)
    fused=base+struct_gain*sd*ss
    order=np.argsort(-fused);ranked=[candidates[i] for i in order]
    return ranked.index(true_answer)+1,{candidates[i]:float(fused[i]) for i in range(len(candidates))}

def v45_probe_panel(model,vocab,chains,seed,variant,bank=None,parity=True):
    rows=[];q=chains[chains.kind=='probe'].reset_index(drop=True);zcands=q.z.tolist()
    def add(kind,prompt,ans,item,mode='plain',candidates=None,wrong=None):
        aug=prompt;retr=[];logs=[];struct={};paths=[];meta={}
        if mode=='text_assist' and bank is not None:
            retr,logs=bank.retrieve_templates(prompt,3,parity);aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        elif mode=='second_only' and bank is not None:
            rr,ll=bank.retrieve_templates(prompt,3,parity);retr=rr[-1:] if rr else [];logs=ll
            aug=(' '.join(retr)+' '+prompt).strip() if retr else prompt
        elif mode in ('identity_join','identity_shuffle') and bank is not None and hasattr(bank,'identity_join'):
            struct,paths,meta=bank.identity_join(prompt,parity,mode=='identity_shuffle')
        r=token_rank(model,vocab,aug,ans);cr=np.nan;margin=np.nan
        if candidates:
            if mode in ('identity_join','identity_shuffle'):
                cr,sc=fused_candidate_rank(model,vocab,aug,ans,candidates,struct)
            else:
                cr,sc=candidate_rank(model,vocab,aug,ans,candidates)
            ts=sc.get(ans,np.nan);ws=sc.get(wrong,np.nan);margin=ts-ws
        else:ts=ws=np.nan
        rows.append({'seed':seed,'variant':variant,'probe':kind,'mode':mode,'item':item,'answer':ans,
                     'rank':r,'candidate_rank':cr,'binding_margin':margin,
                     'candidate_top1':int(cr==1) if np.isfinite(cr) else np.nan,
                     'retrieved':' || '.join(retr),'join_resolved':bool(meta.get('resolved',False)),
                     'join_paths':json.dumps(paths),'struct_true_score':float(struct.get(ans,0.)) if candidates else np.nan,
                     'struct_wrong_score':float(struct.get(wrong,0.)) if candidates and wrong else np.nan})
    for i,(p,a) in enumerate(FACTS):add('exact_fact',p,a,f'f{i}')
    for i,(p,a) in enumerate(REPHRASE):add('rephrase',p,a,f'r{i}')
    for i,row in q.iterrows():
        wrong=q.iloc[(i+1)%len(q)]
        for mode in ['plain','text_assist','second_only','identity_join','identity_shuffle']:
            add('two_hop',f'{row.x} ultimately maps to',row.z,f'c{i}',mode,zcands,wrong.z)
        add('premise_contains',f'{row.x} contains',row.y,f'c{i}')
        add('premise_maps',f'{row.y} maps to',row.z,f'c{i}')
    return pd.DataFrame(rows)

V45_VARIANTS=['baseline','v43_factor_gate','v44_multi_auto','v45_multi_access','v45_pareto_access','v45_structural_graph']

def run_one_v45(seed=0,full=False,corpus='interleaved',device=None,capacity=96,contextual_demo=False):
    seed_all(seed);train,chains=make_training_corpus(seed,contextual_demo=contextual_demo)
    stream=natural_stream(seed,corpus,700) if full else proxy_stream(seed,180);vocab=Vocab(train+stream)
    if device is None:device='cuda' if full and torch.cuda.is_available() else 'cpu'

    base=TinySSM(len(vocab.itos)).to(device);opt=torch.optim.Adam(base.parameters(),lr=.006);order=list(range(len(train)))
    early_state=mid_state=None
    for ep in range(3):
        random.Random(seed+ep).shuffle(order)
        for i in order:train_sentence(base,opt,vocab.encode(train[i]))
        if ep==0:early_state=copy.deepcopy(base.state_dict())
        if ep==1:mid_state=copy.deepcopy(base.state_dict())
    final_state=copy.deepcopy(base.state_dict())
    early=TinySSM(len(vocab.itos)).to(device);early.load_state_dict(early_state)
    mid=TinySSM(len(vocab.itos)).to(device);mid.load_state_dict(mid_state)
    cand=add_stability(unique_candidate_table(train,vocab,early,mid,base))
    cand_v43,meta_v43=evidence_model(cand,'factor');cand_multi,meta_multi=multi_consequence_model(cand,'factor')
    v45bank,merged=build_multi_access_bank(cand_v43,cand_multi,vocab,capacity,True)
    paretobank,_=build_pareto_multi_access_bank(cand_v43,cand_multi,vocab,capacity)
    graphbank=GraphBindingSystem(build_v44_bank(cand_multi,vocab,'multi_auto',capacity),IdentityBindingGraph(cand_v43,512))
    banks={'baseline':None,'v43_factor_gate':build_bank(cand_v43,meta_v43,vocab,'factor_gate',capacity),
           'v44_multi_auto':build_v44_bank(cand_multi,vocab,'multi_auto',capacity),
           'v45_multi_access':v45bank,'v45_pareto_access':paretobank,'v45_structural_graph':graphbank}

    runs=[];probes=[];audit=[]
    qp=chains[chains.kind=='probe'].reset_index(drop=True);prem=set()
    for _,r in qp.iterrows():prem|={f'{r.x} contains {r.y}.',f'{r.y} maps to {r.z}.'}
    for variant in V45_VARIANTS:
        model=TinySSM(len(vocab.itos)).to(device);model.load_state_dict(final_state);opt=torch.optim.Adam(model.parameters(),lr=.006)
        bank=banks[variant];repair=lost=0;replay=[]
        if bank is not None:
            bank.corrupt(.18,seed+1234);replay,repair,lost=bank.replay_ids(True)
        total=0
        for ep in range(2 if full else 1):
            for s in stream:
                train_sentence(model,opt,vocab.encode(s));total+=1
                if replay and total%6==0:train_sentence(model,opt,replay[(total//6)%len(replay)])
        post=v45_probe_panel(model,vocab,chains,seed,variant,bank,True);probes.append(post)
        def mrr(k,m='plain'):
            x=post[(post.probe==k)&(post['mode']==m)]['rank'].to_numpy(float);return float(np.mean(1/x)) if len(x) else np.nan
        def cmrr(m):
            x=post[(post.probe=='two_hop')&(post['mode']==m)].candidate_rank.to_numpy(float);return float(np.nanmean(1/x))
        def bmar(m):
            return float(np.nanmean(post[(post.probe=='two_hop')&(post['mode']==m)].binding_margin))
        jq=post[(post.probe=='two_hop')&(post['mode']=='identity_join')]
        if bank is not None:
            stored={s.text for s in bank.slots}
            both=np.mean([f'{r.x} contains {r.y}.' in stored and f'{r.y} maps to {r.z}.' in stored for _,r in qp.iterrows()])
            audit.append({'seed':seed,'variant':variant,'slots':len(bank.slots),'writes':bank.writes,'replacements':bank.replacements,
                          'blocked':bank.blocked,'holds':bank.holds,'premise_template_coverage':len(stored&prem)/len(prem),
                          'both_premises_stored':both,'repair':repair,'lost':lost})
        runs.append({'seed':seed,'variant':variant,'exact_mrr':mrr('exact_fact'),'rephrase_mrr':mrr('rephrase'),
                     'premise_contains_mrr':mrr('premise_contains'),'premise_maps_mrr':mrr('premise_maps'),
                     'plain_candidate_mrr':cmrr('plain'),'text_assist_candidate_mrr':cmrr('text_assist'),
                     'second_only_candidate_mrr':cmrr('second_only'),'identity_join_candidate_mrr':cmrr('identity_join'),
                     'identity_shuffle_candidate_mrr':cmrr('identity_shuffle'),
                     'identity_join_binding_margin':bmar('identity_join'),'identity_shuffle_binding_margin':bmar('identity_shuffle'),
                     'identity_gain_over_best_nonjoin':cmrr('identity_join')-max(cmrr('plain'),cmrr('text_assist'),cmrr('second_only')),
                     'identity_control_gap':cmrr('identity_join')-cmrr('identity_shuffle'),
                     'join_resolved_rate':float(jq.join_resolved.mean()),'join_true_struct_support':float(jq.struct_true_score.mean()),
                     'join_wrong_struct_support':float(jq.struct_wrong_score.mean()),'parity_repaired':repair,'parity_lost':lost,
                     'stream_loss':float(np.mean([seq_loss(model,vocab.encode(s)) for s in stream[-60:]]))})
    return pd.DataFrame(runs),pd.concat(probes,ignore_index=True),pd.DataFrame(audit),merged,{'v43':meta_v43,'v44':meta_multi}

def main(argv=None):
    ap=argparse.ArgumentParser();ap.add_argument('--full',action='store_true')
    ap.add_argument('--corpus',choices=['shakespeare','bible','interleaved'],default='interleaved')
    ap.add_argument('--seeds',type=int,default=3);ap.add_argument('--capacity',type=int,default=96)
    ap.add_argument('--contextual-demo',action='store_true');args=ap.parse_args(argv)
    RR=[];PP=[];BB=[];MM=[];EE=[]
    for seed in range(args.seeds):
        print('seed',seed)
        r,p,b,m,e=run_one_v45(seed,args.full,args.corpus,capacity=args.capacity,contextual_demo=args.contextual_demo)
        RR.append(r);PP.append(p);BB.append(b);m['seed']=seed;MM.append(m);EE.append(e)
        print(r[['variant','exact_mrr','identity_join_candidate_mrr','identity_join_binding_margin','identity_gain_over_best_nonjoin']].to_string(index=False))
    runs=pd.concat(RR,ignore_index=True);probes=pd.concat(PP,ignore_index=True);banks=pd.concat(BB,ignore_index=True);merged=pd.concat(MM,ignore_index=True)
    tag=(('full_'+args.corpus) if args.full else 'proxy')+('_contextual' if args.contextual_demo else '_legacy')
    runs.to_csv(OUT/f'{tag}_{args.seeds}seed_runs.csv',index=False);probes.to_csv(OUT/f'{tag}_{args.seeds}seed_probe_details.csv',index=False)
    banks.to_csv(OUT/f'{tag}_{args.seeds}seed_bank_audit.csv',index=False);merged.to_csv(OUT/f'{tag}_{args.seeds}seed_multiaccess_values.csv',index=False)
    (OUT/f'{tag}_{args.seeds}seed_evidence_models.json').write_text(json.dumps(EE,indent=2))
    def vals(v,c):return runs[runs.variant==v].sort_values('seed')[c].to_numpy(float)
    checks={'v45_identity_join_margin_positive_seed_fraction':float(np.mean(vals('v45_multi_access','identity_join_binding_margin')>0)),
            'v45_identity_gain_positive_seed_fraction':float(np.mean(vals('v45_multi_access','identity_gain_over_best_nonjoin')>0)),
            'v45_identity_control_gap_positive_seed_fraction':float(np.mean(vals('v45_multi_access','identity_control_gap')>0)),
            'v45_join_resolved_rate_median':float(np.median(vals('v45_multi_access','join_resolved_rate'))),
            'v45_exact_beats_v43_seed_fraction':float(np.mean(vals('v45_multi_access','exact_mrr')>vals('v43_factor_gate','exact_mrr'))),
            'v45_exact_beats_v44_auto_seed_fraction':float(np.mean(vals('v45_multi_access','exact_mrr')>vals('v44_multi_auto','exact_mrr'))),
            'v45_pareto_identity_join_margin_positive_seed_fraction':float(np.mean(vals('v45_pareto_access','identity_join_binding_margin')>0)),
            'v45_pareto_identity_gain_positive_seed_fraction':float(np.mean(vals('v45_pareto_access','identity_gain_over_best_nonjoin')>0)),
            'v45_pareto_identity_control_gap_positive_seed_fraction':float(np.mean(vals('v45_pareto_access','identity_control_gap')>0)),
            'v45_pareto_join_resolved_rate_median':float(np.median(vals('v45_pareto_access','join_resolved_rate'))),
            'v45_graph_binding_margin_positive_seed_fraction':float(np.mean(vals('v45_structural_graph','identity_join_binding_margin')>0)),
            'v45_graph_identity_gain_positive_seed_fraction':float(np.mean(vals('v45_structural_graph','identity_gain_over_best_nonjoin')>0)),
            'v45_graph_control_gap_positive_seed_fraction':float(np.mean(vals('v45_structural_graph','identity_control_gap')>0)),
            'v45_graph_join_resolved_rate_median':float(np.median(vals('v45_structural_graph','join_resolved_rate'))),
            'parity_repairs_faults':bool(np.median(vals('v45_structural_graph','parity_repaired'))>0)}
    (OUT/f'{tag}_{args.seeds}seed_checks.json').write_text(json.dumps(checks,indent=2))
    med=runs.groupby('variant').median(numeric_only=True);bankmed=banks.groupby('variant').median(numeric_only=True)
    report=f"""Adrianic v4.5 — Multi-Access Memory + Identity Binding State

Mode: {tag}
Seeds: {args.seeds}
Capacity: {args.capacity}
Contextual demonstration ingestion: {args.contextual_demo}

Each memory object has independent consolidation/replay value C and retrieval/address value R.
Replay uses C; query addressing uses R. Resource survival may be earned by either role.

The identity-join operator knows no relation names, semantic roles, chain IDs, target labels or answer keys.
It uses rare query anchors, exact shared-token identity across two templates, observed sequence order and learned retrieval access.

The composed X->Z target sentence remains absent for every probe chain.

Median metrics
==============
{med.to_string()}

Median bank audit
=================
{bankmed.to_string()}

Checks
======
{json.dumps(checks,indent=2)}

Interpretation boundary
=======================
A positive identity-join result is SYSTEM-level deterministic compositional binding.
It is not evidence that the RNN independently induced first-order logic internally.

Critical controls:
- identity join must beat plain/text/second-only;
- identity join should beat identity-shuffled support;
- no probe composed target is present in training;
- replay and retrieval values remain separately logged.
"""
    (OUT/f'{tag}_{args.seeds}seed_REPORT.txt').write_text(report);print(report)



# =============================================================================
# Adrianic v4.6 — Variable-Depth Identity-Path Stress Test
#
# Question:
#   Does the structural identity bridge remain useful at depths 2, 3 and 4
#   after interference, or does it collapse into a two-hop special case?
#
# The test keeps the same v4.4 multi-consequence replay learner and adds only
# a generic N-step traversal over recurrence-qualified token-identity edges.
#
# Boundaries:
#   - This is SYSTEM-level structural composition.
#   - It is not evidence that the RNN internally discovered logic.
#   - The traversal depth is supplied by the query/test condition.
# =============================================================================

OUT46 = ROOT / "adrianic_v46_depth_stress_results"
OUT46.mkdir(parents=True, exist_ok=True)

def make_deep_chains(seed=0, depths=(2,3,4), n_demo=10, n_probe=8):
    """Create disjoint variable-depth chains.

    A depth-D chain has D observed edges and D+1 unique entity tokens.
    Probe-chain final target sentences are never included.
    """
    rng = random.Random(seed + 4600)
    total_nodes = sum((n_demo + n_probe) * (d + 1) for d in depths)
    names = unique_words(total_nodes, seed + 4617)
    cursor = 0
    train = []
    rows = []

    for depth in depths:
        for idx in range(n_demo + n_probe):
            nodes = names[cursor:cursor + depth + 1]
            cursor += depth + 1
            kind = "demo" if idx < n_demo else "probe"

            premises = [
                f"{nodes[j]} links to {nodes[j+1]}."
                for j in range(depth)
            ]

            if kind == "demo":
                # Demonstrates the language form for a depth-specific conclusion,
                # but only for disjoint demonstration entities.
                target = f"{nodes[0]} reaches{depth} {nodes[-1]}."
                train += (premises + [target]) * 3
                # Contiguous context is included only for demonstrations.
                paragraph = " ".join(premises + [target])
                train += [paragraph, paragraph]
            else:
                # Probe entities receive only their individual observed edges.
                train += premises * 4

            rows.append({
                "kind": kind,
                "depth": depth,
                "chain_id": f"d{depth}_{idx}",
                "nodes": nodes,
                "source": nodes[0],
                "target": nodes[-1],
                "premises": premises,
            })

    # Keep the ordinary factual substrate and one-off distractors.
    for p, a in FACTS:
        train += [f"{p} {a}.", f"fact: {p} {a}.", f"remember {p} {a}."]

    for i in range(90):
        train.append(f"depthnoise{i} fog{i} copper{i} lantern{i} stray{i}.")

    rng.shuffle(train)
    return train, pd.DataFrame(rows)


class VariableDepthIdentityGraph(IdentityBindingGraph):
    """Generic exact-identity path traversal for an arbitrary supplied depth."""

    def n_step(self, prompt, depth):
        x = self._anchor(prompt)
        if x is None or depth < 1:
            return {}, []

        frontier = {x: 1.0}
        paths = [([x], 1.0)]

        for _ in range(depth):
            new_frontier = defaultdict(float)
            new_paths = []
            for path, path_score in paths:
                node = path[-1]
                for nxt, weight in self.out.get(node, {}).items():
                    if nxt in path:
                        continue
                    # Geometric accumulation prevents one huge edge from masking
                    # a weak edge elsewhere in the path.
                    new_score = (path_score * max(weight, 1e-12)) ** (1.0 / 2.0)
                    new_frontier[nxt] += new_score
                    new_paths.append((path + [nxt], new_score))
            paths = new_paths
            frontier = new_frontier
            if not paths:
                break

        raw = defaultdict(float)
        for path, score in paths:
            if len(path) == depth + 1:
                raw[path[-1]] += score

        mx = max(raw.values(), default=0.0) + 1e-12
        norm = {k: float(v / mx) for k, v in raw.items()}
        path_log = [
            {"path": path, "score": float(score)}
            for path, score in sorted(paths, key=lambda x: x[1], reverse=True)[:20]
            if len(path) == depth + 1
        ]
        return norm, path_log

    def shuffled_n_step_mean(self, prompt, depth, n=32):
        """Destroy exact intermediate identity while preserving adjacency pools.

        At each hop, the outgoing adjacency list is borrowed from a deterministically
        shuffled source node. First-hop anchor and global edge-weight distributions
        remain available, but bridge identity is broken.
        """
        x = self._anchor(prompt)
        if x is None or depth < 1:
            return {}

        source_nodes = sorted(self.out.keys())
        if len(source_nodes) < 2:
            return {}

        base = int(hashlib.sha256(f"{prompt}|{depth}".encode()).hexdigest()[:12], 16)
        acc = defaultdict(float)

        for rep in range(n):
            rng = random.Random(base + rep * 104729)
            permutations = []
            for hop in range(depth):
                perm = list(source_nodes)
                rng.shuffle(perm)
                permutations.append(dict(zip(source_nodes, perm)))

            paths = [([x], 1.0)]
            for hop in range(depth):
                next_paths = []
                mapping = permutations[hop]
                for path, path_score in paths:
                    node = path[-1]
                    donor = node if hop == 0 else mapping.get(node, node)
                    for nxt, weight in self.out.get(donor, {}).items():
                        if nxt in path:
                            continue
                        new_score = (path_score * max(weight, 1e-12)) ** 0.5
                        next_paths.append((path + [nxt], new_score))
                paths = next_paths
                if not paths:
                    break

            raw = defaultdict(float)
            for path, score in paths:
                if len(path) == depth + 1:
                    raw[path[-1]] += score
            mx = max(raw.values(), default=0.0) + 1e-12
            for token, value in raw.items():
                acc[token] += float(value / mx) / n

        return dict(acc)


def make_v46_models(seed, train, stream, device):
    vocab = Vocab(train + stream)
    seed_all(seed)

    base = TinySSM(len(vocab.itos)).to(device)
    opt = torch.optim.Adam(base.parameters(), lr=.006)
    order = list(range(len(train)))
    early_state = mid_state = None

    for epoch in range(3):
        random.Random(seed + epoch).shuffle(order)
        for i in order:
            train_sentence(base, opt, vocab.encode(train[i]))
        if epoch == 0:
            early_state = copy.deepcopy(base.state_dict())
        if epoch == 1:
            mid_state = copy.deepcopy(base.state_dict())

    final_state = copy.deepcopy(base.state_dict())

    early = TinySSM(len(vocab.itos)).to(device)
    early.load_state_dict(early_state)
    mid = TinySSM(len(vocab.itos)).to(device)
    mid.load_state_dict(mid_state)

    cand = add_stability(unique_candidate_table(train, vocab, early, mid, base))
    cand_v43, meta_v43 = evidence_model(cand, "factor")
    cand_multi, meta_multi = multi_consequence_model(cand, "factor")

    replay_bank = build_v44_bank(cand_multi, vocab, "multi_auto", 96)
    graph = VariableDepthIdentityGraph(cand_v43, max_edges=1024)

    return vocab, final_state, replay_bank, graph, cand_v43, cand_multi, {
        "v43": meta_v43,
        "v44": meta_multi,
        "edge_count": len(graph.edges),
    }


def score_depth_panel(model, vocab, graph, chain_df, seed):
    rows = []

    for depth in sorted(chain_df.depth.unique()):
        probes = chain_df[
            (chain_df.kind == "probe") & (chain_df.depth == depth)
        ].reset_index(drop=True)

        candidates = probes.target.tolist()

        for i, row in probes.iterrows():
            wrong = probes.iloc[(i + 1) % len(probes)].target
            prompt = f"{row.source} reaches{depth}"

            base_rank, base_scores = candidate_rank(
                model, vocab, prompt, row.target, candidates
            )

            exact_struct, exact_paths = graph.n_step(prompt, depth)
            exact_rank, exact_scores = fused_candidate_rank(
                model, vocab, prompt, row.target, candidates,
                exact_struct, struct_gain=1.35
            )

            shuffle_struct = graph.shuffled_n_step_mean(
                prompt, depth, n=32
            )
            shuffle_rank, shuffle_scores = fused_candidate_rank(
                model, vocab, prompt, row.target, candidates,
                shuffle_struct, struct_gain=1.35
            )

            # Under-depth control: one fewer hop cannot reach the intended
            # terminal through the full chain unless an illicit shortcut exists.
            short_struct, short_paths = graph.n_step(
                prompt, max(1, depth - 1)
            )
            short_rank, short_scores = fused_candidate_rank(
                model, vocab, prompt, row.target, candidates,
                short_struct, struct_gain=1.35
            )

            rows.append({
                "seed": seed,
                "depth": depth,
                "chain_id": row.chain_id,
                "source": row.source,
                "target": row.target,
                "wrong": wrong,
                "plain_candidate_rank": base_rank,
                "identity_candidate_rank": exact_rank,
                "shuffle_candidate_rank": shuffle_rank,
                "underdepth_candidate_rank": short_rank,
                "identity_binding_margin":
                    exact_scores[row.target] - exact_scores[wrong],
                "shuffle_binding_margin":
                    shuffle_scores[row.target] - shuffle_scores[wrong],
                "underdepth_binding_margin":
                    short_scores[row.target] - short_scores[wrong],
                "identity_control_gap_mrr":
                    1.0 / exact_rank - 1.0 / shuffle_rank,
                "identity_underdepth_gap_mrr":
                    1.0 / exact_rank - 1.0 / short_rank,
                "join_resolved": bool(exact_struct),
                "target_struct_support": float(
                    exact_struct.get(row.target, 0.0)
                ),
                "wrong_struct_support": float(
                    exact_struct.get(wrong, 0.0)
                ),
                "paths": json.dumps(exact_paths),
                "short_paths": json.dumps(short_paths),
            })

    return pd.DataFrame(rows)


def run_one_v46(seed=0, full=False, corpus="interleaved"):
    train, chains = make_deep_chains(seed)
    stream = (
        natural_stream(seed, corpus, 700)
        if full else proxy_stream(seed, 180)
    )
    device = "cuda" if full and torch.cuda.is_available() else "cpu"

    vocab, final_state, replay_bank, graph, cand_v43, cand_multi, meta = (
        make_v46_models(seed, train, stream, device)
    )

    model = TinySSM(len(vocab.itos)).to(device)
    model.load_state_dict(final_state)
    opt = torch.optim.Adam(model.parameters(), lr=.006)

    replay_bank.corrupt(.18, seed + 46000)
    replay, repaired, lost = replay_bank.replay_ids(True)

    total = 0
    for epoch in range(2 if full else 1):
        for sentence in stream:
            train_sentence(model, opt, vocab.encode(sentence))
            total += 1
            if replay and total % 6 == 0:
                train_sentence(
                    model, opt,
                    replay[(total // 6) % len(replay)]
                )

    panel = score_depth_panel(model, vocab, graph, chains, seed)
    panel["parity_repaired"] = repaired
    panel["parity_lost"] = lost
    panel["stream_loss"] = float(np.mean([
        seq_loss(model, vocab.encode(s))
        for s in stream[-60:]
    ]))

    summaries = []
    for depth, group in panel.groupby("depth"):
        summaries.append({
            "seed": seed,
            "depth": int(depth),
            "plain_mrr": float(np.mean(
                1.0 / group.plain_candidate_rank
            )),
            "identity_mrr": float(np.mean(
                1.0 / group.identity_candidate_rank
            )),
            "shuffle_mrr": float(np.mean(
                1.0 / group.shuffle_candidate_rank
            )),
            "underdepth_mrr": float(np.mean(
                1.0 / group.underdepth_candidate_rank
            )),
            "identity_binding_margin": float(
                group.identity_binding_margin.mean()
            ),
            "shuffle_binding_margin": float(
                group.shuffle_binding_margin.mean()
            ),
            "underdepth_binding_margin": float(
                group.underdepth_binding_margin.mean()
            ),
            "strict_gain_over_plain": float(np.mean(
                1.0 / group.identity_candidate_rank
                - 1.0 / group.plain_candidate_rank
            )),
            "identity_control_gap": float(
                group.identity_control_gap_mrr.mean()
            ),
            "identity_underdepth_gap": float(
                group.identity_underdepth_gap_mrr.mean()
            ),
            "join_resolved_rate": float(
                group.join_resolved.mean()
            ),
            "target_struct_support": float(
                group.target_struct_support.mean()
            ),
            "wrong_struct_support": float(
                group.wrong_struct_support.mean()
            ),
            "edge_count": len(graph.edges),
            "parity_repaired": repaired,
            "parity_lost": lost,
            "stream_loss": float(panel.stream_loss.iloc[0]),
        })

    return (
        pd.DataFrame(summaries),
        panel,
        cand_v43,
        cand_multi,
        meta,
    )


def main46(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--full", action="store_true")
    parser.add_argument(
        "--corpus",
        choices=["shakespeare", "bible", "interleaved"],
        default="interleaved",
    )
    parser.add_argument("--seeds", type=int, default=3)
    args = parser.parse_args(argv)

    summaries = []
    details = []
    metas = []

    for seed in range(args.seeds):
        print("seed", seed)
        summary, detail, _, _, meta = run_one_v46(
            seed=seed,
            full=args.full,
            corpus=args.corpus,
        )
        summaries.append(summary)
        details.append(detail)
        meta["seed"] = seed
        metas.append(meta)
        print(summary[
            [
                "depth",
                "identity_mrr",
                "shuffle_mrr",
                "underdepth_mrr",
                "identity_binding_margin",
                "strict_gain_over_plain",
                "identity_control_gap",
            ]
        ].to_string(index=False))

    summary = pd.concat(summaries, ignore_index=True)
    detail = pd.concat(details, ignore_index=True)

    tag = (
        f"full_{args.corpus}"
        if args.full else "proxy"
    )

    summary.to_csv(
        OUT46 / f"{tag}_{args.seeds}seed_depth_summary.csv",
        index=False,
    )
    detail.to_csv(
        OUT46 / f"{tag}_{args.seeds}seed_probe_details.csv",
        index=False,
    )
    (OUT46 / f"{tag}_{args.seeds}seed_models.json").write_text(
        json.dumps(metas, indent=2)
    )

    checks = {}
    for depth in sorted(summary.depth.unique()):
        g = summary[summary.depth == depth]
        checks[f"d{depth}_margin_positive_seed_fraction"] = float(
            np.mean(g.identity_binding_margin > 0)
        )
        checks[f"d{depth}_strict_gain_positive_seed_fraction"] = float(
            np.mean(g.strict_gain_over_plain > 0)
        )
        checks[f"d{depth}_shuffle_gap_positive_seed_fraction"] = float(
            np.mean(g.identity_control_gap > 0)
        )
        checks[f"d{depth}_underdepth_gap_positive_seed_fraction"] = float(
            np.mean(g.identity_underdepth_gap > 0)
        )
        checks[f"d{depth}_join_resolved_median"] = float(
            np.median(g.join_resolved_rate)
        )

    checks["parity_repairs_faults"] = bool(
        summary.parity_repaired.median() > 0
    )
    checks["no_parity_losses"] = bool(
        summary.parity_lost.max() == 0
    )

    (OUT46 / f"{tag}_{args.seeds}seed_checks.json").write_text(
        json.dumps(checks, indent=2)
    )

    med = summary.groupby("depth").median(numeric_only=True)

    report = f"""Adrianic v4.6 — Variable-Depth Identity-Path Stress Test

Mode: {tag}
Seeds: {args.seeds}

Question
========
Does exact-identity structural composition remain useful at depths 2, 3 and 4?

Training boundary
=================
For every probe chain, only the individual adjacent edges are present.
The final source->target conclusion is absent.
Demonstration and probe entities are disjoint.

Structural operator
===================
The same generic N-step operator is used at every depth.
It follows recurrence-qualified directed token-identity edges.
It knows no relation names, chain IDs, answer keys or target labels.

Controls
========
- plain recurrent learner with no structural support
- bridge-identity shuffled traversal
- under-depth traversal using one fewer hop

Median metrics by depth
=======================
{med.to_string()}

Checks
======
{json.dumps(checks, indent=2)}

Interpretation boundary
=======================
A positive result is system-level variable-depth identity composition.
It is not evidence that the RNN internally induced formal logic.

Depth is supplied to the traversal for this experiment.
A later experiment must test endogenous stopping / path-length selection.
"""
    (OUT46 / f"{tag}_{args.seeds}seed_REPORT.txt").write_text(report)
    print(report)




# =============================================================================
# Adrianic v4.7 — Long-Horizon Identity Depth Frontier
#
# Extends the variable-depth stress test to:
#   2, 3, 4, 7, 9, and 15 identity edges.
#
# Adds diagnostics that separate:
#   - required-edge survival,
#   - path resolution,
#   - endpoint discrimination,
#   - under-traversal,
#   - over-traversal,
#   - bridge-identity destruction.
#
# The requested depth is still supplied externally. Endogenous stopping remains
# a later experiment.
# =============================================================================

OUT47 = ROOT / "adrianic_v47_long_depth_frontier_results"
OUT47.mkdir(parents=True, exist_ok=True)

V47_DEPTHS = (2, 3, 4, 7, 9, 15)


def _frontier_alpha_token(i):
    # Unlimited opaque alphabetic entity IDs so tokenization keeps each ID intact.
    # Example: qaa, qab, ...  No semantic information is encoded in the ID.
    chars = []
    n = int(i)
    while True:
        chars.append(chr(ord("a") + (n % 26)))
        n = n // 26 - 1
        if n < 0:
            break
    return "q" + "".join(reversed(chars))


def make_frontier_chains(seed=0, depths=V47_DEPTHS, n_demo=10, n_probe=8):
    rng = random.Random(seed + 4700)
    total_nodes = sum(
        (n_demo + n_probe) * (int(depth) + 1)
        for depth in depths
    )

    names = [
        _frontier_alpha_token(i + seed * 10000)
        for i in range(total_nodes)
    ]
    rng.shuffle(names)

    cursor = 0
    train = []
    rows = []

    for depth in depths:
        depth = int(depth)

        for idx in range(n_demo + n_probe):
            nodes = names[cursor:cursor + depth + 1]
            cursor += depth + 1
            kind = "demo" if idx < n_demo else "probe"

            premises = [
                f"{nodes[j]} links to {nodes[j+1]}."
                for j in range(depth)
            ]

            if kind == "demo":
                target = (
                    f"{nodes[0]} reaches{depth} "
                    f"{nodes[-1]}."
                )
                train += (premises + [target]) * 3

                paragraph = " ".join(
                    premises + [target]
                )
                train += [paragraph, paragraph]

            else:
                # No source->terminal conclusion for probe entities.
                train += premises * 4

            rows.append({
                "kind": kind,
                "depth": depth,
                "chain_id": f"d{depth}_{idx}",
                "nodes": nodes,
                "source": nodes[0],
                "target": nodes[-1],
                "premises": premises,
            })

    for p, a in FACTS:
        train += [
            f"{p} {a}.",
            f"fact: {p} {a}.",
            f"remember {p} {a}.",
        ]

    for i in range(90):
        train.append(
            f"frontiernoise{i} fog{i} copper{i} "
            f"lantern{i} stray{i}."
        )

    rng.shuffle(train)
    return train, pd.DataFrame(rows)


class FrontierIdentityGraph(VariableDepthIdentityGraph):
    """Variable-depth traversal with a transparent path-resource ceiling.

    The beam is a computational guardrail, not a target. Diagnostics report
    whether it was ever actually active.
    """

    def __init__(self, cand_v43, max_edges=1024, beam_width=512):
        super().__init__(cand_v43, max_edges=max_edges)
        self.beam_width = int(beam_width)

    def n_step_diag(self, prompt, depth):
        x = self._anchor(prompt)
        if x is None or depth < 1:
            return {}, [], {
                "resolved": False,
                "beam_truncations": 0,
                "expanded_paths": 0,
                "final_paths": 0,
            }

        paths = [([x], 1.0)]
        truncations = 0
        expanded_total = 0

        for _ in range(depth):
            next_paths = []
            for path, path_score in paths:
                node = path[-1]
                for nxt, weight in self.out.get(node, {}).items():
                    if nxt in path:
                        continue
                    # Preserve the v4.6 scoring rule exactly.
                    new_score = (path_score * max(weight, 1e-12)) ** 0.5
                    next_paths.append((path + [nxt], new_score))

            expanded_total += len(next_paths)

            if len(next_paths) > self.beam_width:
                next_paths.sort(key=lambda x: x[1], reverse=True)
                next_paths = next_paths[:self.beam_width]
                truncations += 1

            paths = next_paths
            if not paths:
                break

        raw = defaultdict(float)
        final_paths = []
        for path, score in paths:
            if len(path) == depth + 1:
                raw[path[-1]] += score
                final_paths.append((path, score))

        mx = max(raw.values(), default=0.0) + 1e-12
        norm = {k: float(v / mx) for k, v in raw.items()}

        path_log = [
            {"path": path, "score": float(score)}
            for path, score in sorted(
                final_paths, key=lambda x: x[1], reverse=True
            )[:20]
        ]

        return norm, path_log, {
            "resolved": bool(norm),
            "beam_truncations": int(truncations),
            "expanded_paths": int(expanded_total),
            "final_paths": int(len(final_paths)),
        }

    def n_step(self, prompt, depth):
        scores, paths, _ = self.n_step_diag(prompt, depth)
        return scores, paths

    def shuffled_n_step_mean_diag(self, prompt, depth, n=32):
        x = self._anchor(prompt)
        if x is None or depth < 1:
            return {}, {"beam_truncations": 0}

        source_nodes = sorted(self.out.keys())
        if len(source_nodes) < 2:
            return {}, {"beam_truncations": 0}

        base = int(
            hashlib.sha256(f"{prompt}|{depth}".encode()).hexdigest()[:12],
            16,
        )
        acc = defaultdict(float)
        truncations = 0

        for rep in range(n):
            rng = random.Random(base + rep * 104729)
            permutations = []
            for hop in range(depth):
                perm = list(source_nodes)
                rng.shuffle(perm)
                permutations.append(dict(zip(source_nodes, perm)))

            paths = [([x], 1.0)]

            for hop in range(depth):
                next_paths = []
                mapping = permutations[hop]

                for path, path_score in paths:
                    node = path[-1]
                    # Keep the first hop anchored. Break identity at later hops.
                    donor = node if hop == 0 else mapping.get(node, node)

                    for nxt, weight in self.out.get(donor, {}).items():
                        if nxt in path:
                            continue
                        new_score = (
                            path_score * max(weight, 1e-12)
                        ) ** 0.5
                        next_paths.append((path + [nxt], new_score))

                if len(next_paths) > self.beam_width:
                    next_paths.sort(key=lambda x: x[1], reverse=True)
                    next_paths = next_paths[:self.beam_width]
                    truncations += 1

                paths = next_paths
                if not paths:
                    break

            raw = defaultdict(float)
            for path, score in paths:
                if len(path) == depth + 1:
                    raw[path[-1]] += score

            mx = max(raw.values(), default=0.0) + 1e-12
            for token, value in raw.items():
                acc[token] += float(value / mx) / n

        return dict(acc), {"beam_truncations": int(truncations)}

    def shuffled_n_step_mean(self, prompt, depth, n=32):
        scores, _ = self.shuffled_n_step_mean_diag(prompt, depth, n=n)
        return scores


def make_v47_models(
    seed,
    train,
    stream,
    device,
    edge_cap=1024,
    beam_width=512,
):
    vocab = Vocab(train + stream)
    seed_all(seed)

    base_model = TinySSM(len(vocab.itos)).to(device)
    opt = torch.optim.Adam(base_model.parameters(), lr=.006)
    order = list(range(len(train)))
    early_state = mid_state = None

    for epoch in range(3):
        random.Random(seed + epoch).shuffle(order)
        for i in order:
            train_sentence(base_model, opt, vocab.encode(train[i]))
        if epoch == 0:
            early_state = copy.deepcopy(base_model.state_dict())
        if epoch == 1:
            mid_state = copy.deepcopy(base_model.state_dict())

    final_state = copy.deepcopy(base_model.state_dict())

    early = TinySSM(len(vocab.itos)).to(device)
    early.load_state_dict(early_state)
    mid = TinySSM(len(vocab.itos)).to(device)
    mid.load_state_dict(mid_state)

    cand = add_stability(
        unique_candidate_table(train, vocab, early, mid, base_model)
    )
    cand_v43, meta_v43 = evidence_model(cand, "factor")
    cand_multi, meta_multi = multi_consequence_model(cand, "factor")

    replay_bank = build_v44_bank(cand_multi, vocab, "multi_auto", 96)
    graph = FrontierIdentityGraph(
        cand_v43,
        max_edges=edge_cap,
        beam_width=beam_width,
    )

    return vocab, final_state, replay_bank, graph, cand_v43, cand_multi, {
        "v43": meta_v43,
        "v44": meta_multi,
        "edge_count": len(graph.edges),
        "edge_cap": int(edge_cap),
        "beam_width": int(beam_width),
    }


def required_edge_coverage(graph, row):
    nodes = row.nodes
    required = [
        (nodes[j], nodes[j + 1])
        for j in range(len(nodes) - 1)
    ]
    present = sum(edge in graph.edges for edge in required)
    return present / max(1, len(required))


def score_frontier_panel(model, vocab, graph, chain_df, seed):
    rows = []

    for depth in sorted(chain_df.depth.unique()):
        probes = chain_df[
            (chain_df.kind == "probe") &
            (chain_df.depth == depth)
        ].reset_index(drop=True)

        candidates = probes.target.tolist()

        for i, row in probes.iterrows():
            wrong = probes.iloc[(i + 1) % len(probes)].target
            prompt = f"{row.source} reaches{depth}"

            plain_rank, plain_scores = candidate_rank(
                model, vocab, prompt, row.target, candidates
            )

            identity_struct, identity_paths, identity_meta = (
                graph.n_step_diag(prompt, depth)
            )
            identity_rank, identity_scores = fused_candidate_rank(
                model,
                vocab,
                prompt,
                row.target,
                candidates,
                identity_struct,
                struct_gain=1.35,
            )

            shuffle_struct, shuffle_meta = (
                graph.shuffled_n_step_mean_diag(
                    prompt, depth, n=32
                )
            )
            shuffle_rank, shuffle_scores = fused_candidate_rank(
                model,
                vocab,
                prompt,
                row.target,
                candidates,
                shuffle_struct,
                struct_gain=1.35,
            )

            under_depth = max(1, depth - 1)
            under_struct, under_paths, under_meta = graph.n_step_diag(
                prompt, under_depth
            )
            under_rank, under_scores = fused_candidate_rank(
                model,
                vocab,
                prompt,
                row.target,
                candidates,
                under_struct,
                struct_gain=1.35,
            )

            over_struct, over_paths, over_meta = graph.n_step_diag(
                prompt, depth + 1
            )
            over_rank, over_scores = fused_candidate_rank(
                model,
                vocab,
                prompt,
                row.target,
                candidates,
                over_struct,
                struct_gain=1.35,
            )

            rows.append({
                "seed": seed,
                "depth": int(depth),
                "chain_id": row.chain_id,
                "source": row.source,
                "target": row.target,
                "wrong": wrong,

                "plain_candidate_rank": plain_rank,
                "identity_candidate_rank": identity_rank,
                "shuffle_candidate_rank": shuffle_rank,
                "underdepth_candidate_rank": under_rank,
                "overdepth_candidate_rank": over_rank,

                "identity_binding_margin":
                    identity_scores[row.target] -
                    identity_scores[wrong],
                "shuffle_binding_margin":
                    shuffle_scores[row.target] -
                    shuffle_scores[wrong],
                "underdepth_binding_margin":
                    under_scores[row.target] -
                    under_scores[wrong],
                "overdepth_binding_margin":
                    over_scores[row.target] -
                    over_scores[wrong],

                "strict_gain_over_plain":
                    1.0 / identity_rank -
                    1.0 / plain_rank,
                "identity_control_gap":
                    1.0 / identity_rank -
                    1.0 / shuffle_rank,
                "identity_underdepth_gap":
                    1.0 / identity_rank -
                    1.0 / under_rank,
                "identity_overdepth_gap":
                    1.0 / identity_rank -
                    1.0 / over_rank,

                "join_resolved": bool(identity_struct),
                "target_struct_support": float(
                    identity_struct.get(row.target, 0.0)
                ),
                "wrong_struct_support": float(
                    identity_struct.get(wrong, 0.0)
                ),
                "required_edge_coverage": float(
                    required_edge_coverage(graph, row)
                ),

                "beam_truncations": int(
                    identity_meta["beam_truncations"]
                ),
                "expanded_paths": int(
                    identity_meta["expanded_paths"]
                ),
                "final_paths": int(
                    identity_meta["final_paths"]
                ),
                "shuffle_beam_truncations": int(
                    shuffle_meta["beam_truncations"]
                ),

                "paths": json.dumps(identity_paths),
                "under_paths": json.dumps(under_paths),
                "over_paths": json.dumps(over_paths),
            })

    return pd.DataFrame(rows)


def run_one_v47(
    seed=0,
    full=False,
    corpus="interleaved",
    edge_cap=1024,
    beam_width=512,
):
    train, chains = make_frontier_chains(seed)
    stream = (
        natural_stream(seed, corpus, 700)
        if full
        else proxy_stream(seed, 180)
    )
    device = (
        "cuda"
        if full and torch.cuda.is_available()
        else "cpu"
    )

    (
        vocab,
        final_state,
        replay_bank,
        graph,
        cand_v43,
        cand_multi,
        meta,
    ) = make_v47_models(
        seed,
        train,
        stream,
        device,
        edge_cap=edge_cap,
        beam_width=beam_width,
        proxy_fast=(not full),
    )

    model = TinySSM(len(vocab.itos)).to(device)
    model.load_state_dict(final_state)
    opt = torch.optim.Adam(model.parameters(), lr=.006)

    replay_bank.corrupt(.18, seed + 47000)
    replay, repaired, lost = replay_bank.replay_ids(True)

    total = 0
    for epoch in range(2 if full else 1):
        for sentence in stream:
            train_sentence(model, opt, vocab.encode(sentence))
            total += 1
            if replay and total % 6 == 0:
                train_sentence(
                    model,
                    opt,
                    replay[(total // 6) % len(replay)],
                )

    panel = score_frontier_panel(
        model, vocab, graph, chains, seed
    )
    panel["parity_repaired"] = repaired
    panel["parity_lost"] = lost
    panel["stream_loss"] = float(np.mean([
        seq_loss(model, vocab.encode(s))
        for s in stream[-60:]
    ]))

    summaries = []

    for depth, group in panel.groupby("depth"):
        resolved = group[group.join_resolved]

        summaries.append({
            "seed": seed,
            "depth": int(depth),

            "plain_mrr": float(np.mean(
                1.0 / group.plain_candidate_rank
            )),
            "identity_mrr": float(np.mean(
                1.0 / group.identity_candidate_rank
            )),
            "shuffle_mrr": float(np.mean(
                1.0 / group.shuffle_candidate_rank
            )),
            "underdepth_mrr": float(np.mean(
                1.0 / group.underdepth_candidate_rank
            )),
            "overdepth_mrr": float(np.mean(
                1.0 / group.overdepth_candidate_rank
            )),

            "identity_binding_margin": float(
                group.identity_binding_margin.mean()
            ),
            "strict_gain_over_plain": float(
                group.strict_gain_over_plain.mean()
            ),
            "identity_control_gap": float(
                group.identity_control_gap.mean()
            ),
            "identity_underdepth_gap": float(
                group.identity_underdepth_gap.mean()
            ),
            "identity_overdepth_gap": float(
                group.identity_overdepth_gap.mean()
            ),

            "join_resolved_rate": float(
                group.join_resolved.mean()
            ),
            "resolved_identity_mrr": (
                float(np.mean(
                    1.0 / resolved.identity_candidate_rank
                ))
                if len(resolved)
                else np.nan
            ),
            "resolved_binding_margin": (
                float(resolved.identity_binding_margin.mean())
                if len(resolved)
                else np.nan
            ),

            "target_struct_support": float(
                group.target_struct_support.mean()
            ),
            "wrong_struct_support": float(
                group.wrong_struct_support.mean()
            ),
            "required_edge_coverage": float(
                group.required_edge_coverage.mean()
            ),

            "beam_truncations_mean": float(
                group.beam_truncations.mean()
            ),
            "expanded_paths_mean": float(
                group.expanded_paths.mean()
            ),
            "final_paths_mean": float(
                group.final_paths.mean()
            ),

            "edge_count": len(graph.edges),
            "edge_cap": int(edge_cap),
            "beam_width": int(beam_width),
            "parity_repaired": repaired,
            "parity_lost": lost,
            "stream_loss": float(panel.stream_loss.iloc[0]),
        })

    return (
        pd.DataFrame(summaries),
        panel,
        cand_v43,
        cand_multi,
        meta,
    )


def main47(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--full", action="store_true")
    parser.add_argument(
        "--corpus",
        choices=["shakespeare", "bible", "interleaved"],
        default="interleaved",
    )
    parser.add_argument("--seeds", type=int, default=3)
    parser.add_argument("--edge-cap", type=int, default=1024)
    parser.add_argument("--beam-width", type=int, default=512)
    args = parser.parse_args(argv)

    summaries = []
    details = []
    metas = []

    for seed in range(args.seeds):
        print("seed", seed)
        summary, detail, _, _, meta = run_one_v47(
            seed=seed,
            full=args.full,
            corpus=args.corpus,
            edge_cap=args.edge_cap,
            beam_width=args.beam_width,
            n_demo=args.n_demo,
            n_probe=args.n_probe,
        )
        summaries.append(summary)
        details.append(detail)
        meta["seed"] = seed
        metas.append(meta)

        print(summary[[
            "depth",
            "identity_mrr",
            "shuffle_mrr",
            "underdepth_mrr",
            "overdepth_mrr",
            "identity_binding_margin",
            "strict_gain_over_plain",
            "identity_control_gap",
            "join_resolved_rate",
            "required_edge_coverage",
        ]].to_string(index=False))

    summary = pd.concat(summaries, ignore_index=True)
    detail = pd.concat(details, ignore_index=True)

    tag = (
        f"full_{args.corpus}"
        if args.full
        else "proxy"
    )

    summary.to_csv(
        OUT47 / f"{tag}_{args.seeds}seed_frontier_summary.csv",
        index=False,
    )
    detail.to_csv(
        OUT47 / f"{tag}_{args.seeds}seed_probe_details.csv",
        index=False,
    )
    (OUT47 / f"{tag}_{args.seeds}seed_models.json").write_text(
        json.dumps(metas, indent=2)
    )

    checks = {}

    for depth in sorted(summary.depth.unique()):
        g = summary[summary.depth == depth]

        checks[f"d{depth}_margin_positive_seed_fraction"] = float(
            np.mean(g.identity_binding_margin > 0)
        )
        checks[f"d{depth}_strict_gain_positive_seed_fraction"] = float(
            np.mean(g.strict_gain_over_plain > 0)
        )
        checks[f"d{depth}_shuffle_gap_positive_seed_fraction"] = float(
            np.mean(g.identity_control_gap > 0)
        )
        checks[f"d{depth}_underdepth_gap_positive_seed_fraction"] = float(
            np.mean(g.identity_underdepth_gap > 0)
        )
        checks[f"d{depth}_overdepth_gap_positive_seed_fraction"] = float(
            np.mean(g.identity_overdepth_gap > 0)
        )
        checks[f"d{depth}_join_resolved_median"] = float(
            np.median(g.join_resolved_rate)
        )
        checks[f"d{depth}_edge_coverage_median"] = float(
            np.median(g.required_edge_coverage)
        )
        checks[f"d{depth}_beam_truncation_median"] = float(
            np.median(g.beam_truncations_mean)
        )

    checks["parity_repairs_faults"] = bool(
        summary.parity_repaired.median() > 0
    )
    checks["no_parity_losses"] = bool(
        summary.parity_lost.max() == 0
    )

    (OUT47 / f"{tag}_{args.seeds}seed_checks.json").write_text(
        json.dumps(checks, indent=2)
    )

    med = summary.groupby("depth").median(numeric_only=True)

    report = f"""Adrianic v4.7 — Long-Horizon Identity Depth Frontier

Mode: {tag}
Seeds: {args.seeds}
Depths: {list(V47_DEPTHS)}
Edge cap: {args.edge_cap}
Beam width: {args.beam_width}

Question
========
Where does exact-identity structural composition begin to degrade as path
length grows from 2 to 15 observed relationships?

Training boundary
=================
Probe chains receive adjacent edges only.
The final source->target conclusion is absent.
Demonstration and probe entities are disjoint.

Controls
========
- plain recurrent learner
- exact bridge-identity shuffle
- under-depth traversal (depth-1)
- over-depth traversal (depth+1)

Diagnostics
===========
The run separately reports:
- required-edge coverage
- join resolution
- endpoint discrimination
- conditional performance when a path resolves
- whether the computational beam guardrail was actually active

Median metrics by depth
=======================
{med.to_string()}

Checks
======
{json.dumps(checks, indent=2)}

Interpretation boundary
=======================
A positive result is system-level structural path composition.

The requested traversal depth is still supplied externally.
This experiment does not establish endogenous stopping or internally learned
formal logic.
"""

    (OUT47 / f"{tag}_{args.seeds}seed_REPORT.txt").write_text(
        report
    )
    print(report)




# =============================================================================
# Adrianic v4.8 — Ambiguous Topology + Endogenous Structural Stopping
#
# Two questions:
#   A) Can exact identity choose the right path when the graph branches,
#      cycles, shortcuts, and convergent decoys exist?
#   B) Can the system choose where to stop WITHOUT being told the path depth?
#
# The stopping rule is structural, not semantic:
#   stop where endpoint support is concentrated AND further outgoing structural
#   support drops. No probe target, chain ID, answer key, or target depth is
#   supplied to the selector.
#
# Boundary:
#   This is system-level graph reasoning around a recurrent learner.
#   It is not evidence that the RNN itself induced formal logic.
# =============================================================================

OUT48 = ROOT / "adrianic_v48_ambiguous_endogenous_results"
OUT48.mkdir(parents=True, exist_ok=True)

V48_DEPTHS = (4, 7, 9, 15)


def _v48_token(seed, i):
    # Opaque alphabetic entity identity, tokenizer-safe and semantically empty.
    n = int(seed) * 100000 + int(i)
    chars = []
    while True:
        chars.append(chr(ord("a") + (n % 26)))
        n = n // 26 - 1
        if n < 0:
            break
    return "e" + "".join(reversed(chars))


def make_ambiguous_chains(
    seed=0,
    depths=V48_DEPTHS,
    n_demo=4,
    n_probe=4,
):
    """
    Build three-lane ambiguous graphs.

    True lane:
        T0 -> T1 -> ... -> Td

    Two decoy lanes plus cross-links create:
        branches, dead ends, convergence, cycles, and shortcuts.

    Structural evidence difference:
        true edges recur across early/mid/late phases;
        decoy edges have similar local recurrence but poor temporal persistence.

    Probe conclusion:
        "T0 resolves Td."
    is NEVER included.
    """
    rng = random.Random(seed + 4800)
    specs = []
    token_i = 0

    for depth in depths:
        for idx in range(n_demo + n_probe):
            kind = "demo" if idx < n_demo else "probe"

            true_nodes = [
                _v48_token(seed, token_i + j)
                for j in range(depth + 1)
            ]
            token_i += depth + 1

            decoy_a = [
                true_nodes[0]
            ] + [
                _v48_token(seed, token_i + j)
                for j in range(depth)
            ]
            token_i += depth

            decoy_b = [
                true_nodes[0]
            ] + [
                _v48_token(seed, token_i + j)
                for j in range(depth)
            ]
            token_i += depth

            terminal_noise = _v48_token(seed, token_i)
            token_i += 1

            true_edges = [
                f"{true_nodes[j]} links to {true_nodes[j+1]}."
                for j in range(depth)
            ]

            # One full wrong lane plus local dead-end branches.
            # This still creates full-length alternatives while avoiding a
            # combinatorial training corpus explosion.
            decoy_edges = []
            for j in range(depth):
                decoy_edges += [
                    f"{decoy_a[j]} links to {decoy_a[j+1]}.",
                    f"{true_nodes[j]} links to {decoy_a[j+1]}.",
                    f"{true_nodes[j]} links to {decoy_b[j+1]}.",
                ]

                # Sparse convergence back toward the true lane.
                if j + 1 < depth and j % 3 == 1:
                    decoy_edges.append(
                        f"{decoy_a[j+1]} links to {true_nodes[j+2]}."
                    )

                # Sparse cycles.
                if j >= 1 and j % 4 == 2:
                    decoy_edges.append(
                        f"{decoy_b[j]} links to {true_nodes[j-1]}."
                    )

                # Sparse shortcuts that tempt wrong path lengths.
                if j + 2 <= depth and j % 3 == 0:
                    decoy_edges.append(
                        f"{true_nodes[j]} links to {true_nodes[j+2]}."
                    )

            # Terminal is not a perfect dead end; it has one weak transient edge.
            terminal_edge = (
                f"{true_nodes[-1]} links to {terminal_noise}."
            )

            conclusion = (
                f"{true_nodes[0]} resolves {true_nodes[-1]}."
            )

            specs.append({
                "kind": kind,
                "depth": int(depth),
                "chain_id": f"d{depth}_{idx}",
                "true_nodes": true_nodes,
                "decoy_a": decoy_a,
                "decoy_b": decoy_b,
                "terminal_noise": terminal_noise,
                "true_edges": true_edges,
                "decoy_edges": sorted(set(decoy_edges)),
                "terminal_edge": terminal_edge,
                "source": true_nodes[0],
                "target": true_nodes[-1],
                "decoy_targets": [decoy_a[-1], decoy_b[-1]],
                "conclusion": conclusion,
            })

    # IMPORTANT: preserve phase order. Candidate evidence uses temporal span.
    early = []
    mid = []
    late = []

    for spec in specs:
        # True edges exist throughout the experience stream.
        early += spec["true_edges"]
        mid += spec["true_edges"]
        late += spec["true_edges"] * 2

        # Decoys can be locally frequent but are temporally concentrated.
        early += spec["decoy_edges"] * 2

        # Some contradictory/reversed decoy structure appears midstream.
        for sentence in spec["decoy_edges"][::3]:
            toks = content_tokens(sentence)
            if len(toks) >= 2:
                mid.append(f"{toks[-1]} links to {toks[0]}.")

        # Terminal continuation is weak/transient.
        early.append(spec["terminal_edge"])

        # Demonstrations teach only the surface query schema.
        if spec["kind"] == "demo":
            early.append(spec["conclusion"])
            mid.append(spec["conclusion"])
            late.append(spec["conclusion"])

    # Stable factual substrate and unrelated junk.
    facts = []
    for p, a in FACTS:
        facts += [
            f"{p} {a}.",
            f"fact: {p} {a}.",
            f"remember {p} {a}.",
        ]

    junk = [
        f"ambnoise{i} fog{i} copper{i} lantern{i} stray{i}."
        for i in range(120)
    ]

    # Shuffle only WITHIN phase, retaining early/mid/late separation.
    rng.shuffle(early)
    rng.shuffle(mid)
    rng.shuffle(late)
    rng.shuffle(facts)
    rng.shuffle(junk)

    train = (
        early
        + junk[:40]
        + mid
        + facts
        + junk[40:80]
        + late
        + junk[80:]
    )

    return train, pd.DataFrame(specs)


class ConsequenceAmbiguityGraph:
    """
    Sparse identity graph whose edge strength must earn support from TWO
    non-semantic evidence lanes:

      - predictive structural value (v43-style)
      - multi-consequence value (v44-style)

    No relation semantics or target labels enter edge construction.
    """

    def __init__(
        self,
        cand_v43,
        cand_multi,
        max_edges=4096,
        beam_width=512,
    ):
        a = cand_v43.sort_values("uid").reset_index(drop=True)
        b = cand_multi.sort_values("uid").reset_index(drop=True)
        assert np.array_equal(
            a.uid.to_numpy(),
            b.uid.to_numpy(),
        )

        self.idf = compute_idf(a)

        # v4.8 correction:
        # Branch points become frequent BY DEFINITION. A high-IDF gate therefore
        # destroys exactly the identities we are trying to reason through.
        #
        # Instead, exclude only globally ubiquitous glue tokens using document
        # frequency. This is statistical, not semantic: frequent graph hubs are
        # allowed to remain identities.
        docs = [set(content_tokens(s)) for s in a.text]
        df = Counter()
        for d in docs:
            df.update(d)
        Ndocs = max(1, len(docs))
        self.identity_ok = {
            t: (c / Ndocs) <= 0.08
            for t, c in df.items()
        }

        edge_raw = defaultdict(float)
        edge_support = defaultdict(float)
        edge_sources = defaultdict(int)

        for i in range(len(a)):
            ra = a.iloc[i]
            rb = b.iloc[i]

            toks = content_tokens(ra.text)
            identity_tokens = [
                t for t in toks
                if self.identity_ok.get(t, True)
            ]

            clean = []
            for t in identity_tokens:
                if not clean or clean[-1] != t:
                    clean.append(t)

            if len(clean) < 2:
                continue

            # Both evidence lanes must support the sentence.
            u = math.sqrt(
                max(1e-9, float(ra.utility01))
                * max(1e-9, float(rb.utility01))
            )
            q = math.sqrt(
                max(1e-9, float(ra.evidence_confidence))
                * max(1e-9, float(rb.evidence_confidence))
            )
            base = (
                u
                * (.90 + .10 * q)
                * math.log1p(max(1, int(ra["count"])))
            )

            for x in range(len(clean)):
                for y in range(x + 1, len(clean)):
                    s, t = clean[x], clean[y]
                    if s == t:
                        continue
                    edge_raw[(s, t)] += base
                    edge_support[(s, t)] += int(ra["count"])
                    edge_sources[(s, t)] += 1

        edges = [
            (edge, weight, edge_support[edge], edge_sources[edge])
            for edge, weight in edge_raw.items()
            if edge_support[edge] >= 2
        ]
        edges.sort(
            key=lambda x: (x[1], x[2], x[3]),
            reverse=True,
        )
        edges = edges[:max_edges]

        self.edges = {
            edge: float(weight)
            for edge, weight, _, _ in edges
        }
        self.support = {
            edge: float(support)
            for edge, _, support, _ in edges
        }
        self.out = defaultdict(dict)

        weights = np.array(
            list(self.edges.values()),
            float,
        )
        scale = (
            float(np.quantile(weights, .90))
            if len(weights)
            else 1.0
        )
        scale = max(scale, 1e-9)

        for (s, t), w in self.edges.items():
            self.out[s][t] = {
                "raw": float(w),
                "abs": float(min(1.0, w / scale)),
            }

        # Outgoing-relative evidence remains distinct from absolute quality.
        for s, children in self.out.items():
            mx = max(v["raw"] for v in children.values())
            for t in children:
                children[t]["rel"] = float(
                    children[t]["raw"] / (mx + 1e-12)
                )
                children[t]["strength"] = float(
                    math.sqrt(
                        children[t]["abs"]
                        * children[t]["rel"]
                    )
                )

        self.max_edges = int(max_edges)
        self.beam_width = int(beam_width)

    def _anchor(self, prompt):
        q = content_tokens(prompt)
        if not q:
            return None
        return max(
            set(q),
            key=lambda t: self.idf.get(t, 1.0),
        )

    def _expand(self, paths):
        next_paths = []

        for path, log_sum in paths:
            node = path[-1]

            for nxt, info in self.out.get(node, {}).items():
                if nxt in path:
                    continue

                strength = max(
                    1e-12,
                    float(info["strength"]),
                )

                next_paths.append(
                    (
                        path + [nxt],
                        log_sum + math.log(strength),
                    )
                )

        if len(next_paths) > self.beam_width:
            # Compare mean log quality, not raw accumulated product.
            next_paths.sort(
                key=lambda x: x[1] / max(1, len(x[0]) - 1),
                reverse=True,
            )
            next_paths = next_paths[:self.beam_width]
            truncated = 1
        else:
            truncated = 0

        return next_paths, truncated

    def depth_state(self, prompt, depth):
        anchor = self._anchor(prompt)

        if anchor is None or depth < 1:
            return {
                "scores": {},
                "raw_scores": {},
                "mean_scores": {},
                "paths": [],
                "resolved": False,
                "beam_truncations": 0,
                "expanded_paths": 0,
                "final_paths": 0,
            }

        paths = [([anchor], 0.0)]
        trunc = 0
        expanded = 0

        for _ in range(depth):
            paths, tr = self._expand(paths)
            trunc += tr
            expanded += len(paths)

            if not paths:
                break

        endpoint = {}
        endpoint_mean = {}

        for path, log_sum in paths:
            if len(path) != depth + 1:
                continue

            # Product support accumulates independent edge constraints and is
            # used for SAME-DEPTH endpoint discrimination.
            product_score = math.exp(log_sum)

            # Mean path quality is length-normalized and is used only for
            # comparing possible stopping depths.
            mean_score = math.exp(
                log_sum / max(1, depth)
            )
            end = path[-1]

            # Use best path rather than summing correlated route copies.
            if product_score > endpoint.get(end, 0.0):
                endpoint[end] = float(product_score)
            if mean_score > endpoint_mean.get(end, 0.0):
                endpoint_mean[end] = float(mean_score)

        mx = max(endpoint.values(), default=0.0) + 1e-12
        norm = {
            k: float(v / mx)
            for k, v in endpoint.items()
        }

        path_log = []
        for path, log_sum in sorted(
            paths,
            key=lambda x: x[1] / max(1, len(x[0]) - 1),
            reverse=True,
        )[:20]:
            if len(path) == depth + 1:
                path_log.append({
                    "path": path,
                    "score": float(
                        math.exp(log_sum / depth)
                    ),
                })

        return {
            "scores": norm,
            "raw_scores": endpoint,
            "mean_scores": endpoint_mean,
            "paths": path_log,
            "resolved": bool(norm),
            "beam_truncations": int(trunc),
            "expanded_paths": int(expanded),
            "final_paths": int(len(endpoint)),
        }

    def shuffled_depth_state(
        self,
        prompt,
        depth,
        n=24,
    ):
        """
        Preserve graph degree/weight pools but destroy exact bridge identity
        after the first hop.
        """
        anchor = self._anchor(prompt)
        if anchor is None:
            return {"scores": {}, "resolved": False}

        sources = sorted(self.out.keys())
        if len(sources) < 2:
            return {"scores": {}, "resolved": False}

        base_seed = int(
            hashlib.sha256(
                f"{prompt}|{depth}|v48".encode()
            ).hexdigest()[:12],
            16,
        )
        acc = defaultdict(float)

        for rep in range(n):
            rng = random.Random(base_seed + rep * 65537)
            maps = []

            for _ in range(depth):
                perm = list(sources)
                rng.shuffle(perm)
                maps.append(dict(zip(sources, perm)))

            paths = [([anchor], 0.0)]

            for hop in range(depth):
                next_paths = []

                for path, log_sum in paths:
                    node = path[-1]
                    donor = (
                        node
                        if hop == 0
                        else maps[hop].get(node, node)
                    )

                    for nxt, info in self.out.get(donor, {}).items():
                        if nxt in path:
                            continue

                        strength = max(
                            1e-12,
                            float(info["strength"]),
                        )
                        next_paths.append(
                            (
                                path + [nxt],
                                log_sum + math.log(strength),
                            )
                        )

                if len(next_paths) > self.beam_width:
                    next_paths.sort(
                        key=lambda x: x[1] /
                        max(1, len(x[0]) - 1),
                        reverse=True,
                    )
                    next_paths = next_paths[:self.beam_width]

                paths = next_paths
                if not paths:
                    break

            endpoint = {}
            for path, log_sum in paths:
                if len(path) != depth + 1:
                    continue
                score = math.exp(log_sum)
                end = path[-1]
                if score > endpoint.get(end, 0.0):
                    endpoint[end] = score

            mx = max(endpoint.values(), default=0.0) + 1e-12
            for end, score in endpoint.items():
                acc[end] += float(score / mx) / n

        return {
            "scores": dict(acc),
            "resolved": bool(acc),
        }

    def outgoing_abs(self, node):
        children = self.out.get(node, {})
        if not children:
            return 0.0
        return float(max(
            info["abs"]
            for info in children.values()
        ))

    def endogenous_stop(
        self,
        prompt,
        max_depth=15,
    ):
        """
        No target labels and no true depth.

        At each depth, compute:
          - endpoint concentration,
          - normalized entropy,
          - best path quality,
          - remaining outgoing structural pressure.

        Stop where:
          supported endpoint clarity is high
          AND continuation pressure is low.

        There is no tuned target-depth threshold: choose the maximum structural
        stop utility over the allowed search horizon.
        """
        states = []
        best = None

        for depth in range(1, max_depth + 1):
            state = self.depth_state(prompt, depth)
            scores = state["mean_scores"]

            if not scores:
                states.append({
                    "depth": depth,
                    "stop_utility": 0.0,
                    "top": None,
                    "resolved": False,
                })
                continue

            ranked = sorted(
                scores.items(),
                key=lambda x: x[1],
                reverse=True,
            )
            top_node, top_score = ranked[0]
            second_score = (
                ranked[1][1]
                if len(ranked) > 1
                else 0.0
            )

            clarity = float(
                max(
                    0.0,
                    (top_score - second_score)
                    / (top_score + 1e-12),
                )
            )

            arr = np.array(
                [v for _, v in ranked],
                float,
            )
            p = arr / (arr.sum() + 1e-12)

            if len(p) <= 1:
                resolution = 1.0
            else:
                h = -float(np.sum(
                    p * np.log(p + 1e-12)
                ))
                resolution = float(
                    1.0 - h / math.log(len(p))
                )

            continuation = self.outgoing_abs(top_node)

            stop_utility = float(
                top_score
                * (0.5 * clarity + 0.5 * resolution)
                * (1.0 - continuation)
            )

            row = {
                "depth": depth,
                "top": top_node,
                "top_score": float(top_score),
                "clarity": clarity,
                "resolution": resolution,
                "continuation": continuation,
                "stop_utility": stop_utility,
                "resolved": True,
            }
            states.append(row)

            if (
                best is None
                or stop_utility > best["stop_utility"]
            ):
                best = row

        return best, states



def fast_candidate_table_v48(
    train,
    vocab,
    early_model,
    mid_model,
    final_model,
):
    """
    Same evidence vocabulary as v4.4, but avoids the O(N^2) pair scan used by
    the older harness.

    The expensive natural-language context consequence is replaced by one
    nearest-future shared-token continuation probe per unique sentence.

    This keeps the v4.8 stress test computationally practical while preserving
    the important distinctions: recurrence, persistence, temporal span,
    prediction gains, future support and local continuation consequence.
    """
    pos = defaultdict(list)
    token_positions = defaultdict(list)

    token_sets = []
    for i, s in enumerate(train):
        pos[s].append(i)
        ts = set(content_tokens(s))
        token_sets.append(ts)
        for t in ts:
            token_positions[t].append(i)

    docs = list(pos.keys())
    N = len(train)
    df = Counter()
    for s in docs:
        df.update(set(content_tokens(s)))

    rows = []

    for idx, s in enumerate(docs):
        toks = content_tokens(s)
        st = set(toks)
        pp = pos[s]
        first = min(pp)
        last = max(pp)

        recurrence = math.log1p(len(pp))
        token_persistence = (
            float(np.mean([
                math.log1p(df[t])
                for t in toks
            ]))
            if toks else 0.0
        )
        cross_context = (
            float(np.mean([
                df[t] >= 3
                for t in toks
            ]))
            if toks else 0.0
        )
        temporal_span = (
            (last - first) / max(1, N - 1)
            if len(pp) > 1 else 0.0
        )

        early_pred = -seq_loss(
            early_model,
            vocab.encode(s),
        )
        mid_pred = -seq_loss(
            mid_model,
            vocab.encode(s),
        )
        final_pred = -seq_loss(
            final_model,
            vocab.encode(s),
        )

        gain_early_mid = mid_pred - early_pred
        gain_mid_final = final_pred - mid_pred

        # Fraction of the sentence's identities that continue to occur later.
        future_flags = []
        for t in st:
            plist = token_positions[t]
            future_flags.append(
                any(p > first for p in plist)
            )
        future_support = (
            float(np.mean(future_flags))
            if future_flags else 0.0
        )

        # Fast temporal-context consequence proxy.
        # Reward identities that recur in later contexts and across a wide
        # temporal span. This intentionally avoids the old O(N^2) continuation
        # scan so the branching stress test remains runnable.
        context_gain = float(
            future_support * temporal_span
        )

        # Novelty = fraction of identities not seen before first observation.
        if st:
            new_flags = []
            for t in st:
                new_flags.append(
                    token_positions[t][0] == first
                )
            novelty = float(np.mean(new_flags))
        else:
            novelty = 0.0

        rows.append({
            "uid": idx,
            "text": s,
            "first_pos": first,
            "last_pos": last,
            "count": len(pp),
            "recurrence": recurrence,
            "token_persistence": token_persistence,
            "cross_context": cross_context,
            "temporal_span": temporal_span,
            "early_predictability": early_pred,
            "mid_predictability": mid_pred,
            "future_predictability": final_pred,
            "gain_early_mid": gain_early_mid,
            "gain_mid_final": gain_mid_final,
            "future_support": future_support,
            "context_gain": context_gain,
            "novelty": novelty,
            "low_interference": 0.0,
        })

    return pd.DataFrame(rows)


def build_v48_models(
    seed,
    train,
    stream,
    device,
    edge_cap=4096,
    beam_width=512,
    proxy_fast=False,
):
    vocab = Vocab(train + stream)
    seed_all(seed)

    base_model = TinySSM(len(vocab.itos)).to(device)
    opt = torch.optim.Adam(
        base_model.parameters(),
        lr=.006,
    )
    order = list(range(len(train)))
    early_state = mid_state = None

    if proxy_fast:
        # Local mechanics screen: one learning pass with three causal snapshots.
        random.Random(seed).shuffle(order)
        n = len(order)
        cut1 = max(1, n // 3)
        cut2 = max(cut1 + 1, 2 * n // 3)

        for k, i in enumerate(order):
            train_sentence(
                base_model,
                opt,
                vocab.encode(train[i]),
            )
            if k + 1 == cut1:
                early_state = copy.deepcopy(
                    base_model.state_dict()
                )
            if k + 1 == cut2:
                mid_state = copy.deepcopy(
                    base_model.state_dict()
                )

        if early_state is None:
            early_state = copy.deepcopy(
                base_model.state_dict()
            )
        if mid_state is None:
            mid_state = copy.deepcopy(
                base_model.state_dict()
            )

    else:
        # Full test preserves the prior three-stage/three-epoch protocol.
        for epoch in range(3):
            random.Random(seed + epoch).shuffle(order)
            for i in order:
                train_sentence(
                    base_model,
                    opt,
                    vocab.encode(train[i]),
                )

            if epoch == 0:
                early_state = copy.deepcopy(
                    base_model.state_dict()
                )
            if epoch == 1:
                mid_state = copy.deepcopy(
                    base_model.state_dict()
                )

    final_state = copy.deepcopy(
        base_model.state_dict()
    )

    early = TinySSM(len(vocab.itos)).to(device)
    early.load_state_dict(early_state)

    mid = TinySSM(len(vocab.itos)).to(device)
    mid.load_state_dict(mid_state)

    cand = add_stability(
        fast_candidate_table_v48(
            train,
            vocab,
            early,
            mid,
            base_model,
        )
    )

    cand_v43, meta_v43 = evidence_model(
        cand,
        "factor",
    )
    cand_multi, meta_multi = multi_consequence_model(
        cand,
        "factor",
    )

    replay_bank = build_v44_bank(
        cand_multi,
        vocab,
        "multi_auto",
        96,
    )

    graph = ConsequenceAmbiguityGraph(
        cand_v43,
        cand_multi,
        max_edges=edge_cap,
        beam_width=beam_width,
    )

    return (
        vocab,
        final_state,
        replay_bank,
        graph,
        cand_v43,
        cand_multi,
        {
            "v43": meta_v43,
            "v44": meta_multi,
            "edge_count": len(graph.edges),
            "edge_cap": int(edge_cap),
            "beam_width": int(beam_width),
        },
    )


def candidate_pool_for_depth(specs, depth):
    g = specs[
        (specs.kind == "probe")
        & (specs.depth == depth)
    ]

    out = []

    for _, row in g.iterrows():
        out.append(row.target)
        out.extend(list(row.decoy_targets))

    return list(dict.fromkeys(out))


def score_v48_panel(
    model,
    vocab,
    graph,
    specs,
    seed,
    max_depth=15,
):
    rows = []

    probes = specs[
        specs.kind == "probe"
    ].reset_index(drop=True)

    for _, row in probes.iterrows():
        depth = int(row.depth)
        candidates = candidate_pool_for_depth(
            specs,
            depth,
        )

        # Matched wrong terminal from the same chain.
        wrong = list(row.decoy_targets)[0]
        prompt = f"{row.source} resolves"

        plain_rank, plain_scores = candidate_rank(
            model,
            vocab,
            prompt,
            row.target,
            candidates,
        )

        oracle_state = graph.depth_state(
            prompt,
            depth,
        )
        oracle_rank, oracle_scores = fused_candidate_rank(
            model,
            vocab,
            prompt,
            row.target,
            candidates,
            oracle_state["scores"],
            struct_gain=1.35,
        )

        shuffled = graph.shuffled_depth_state(
            prompt,
            depth,
            n=24,
        )
        shuffled_rank, shuffled_scores = fused_candidate_rank(
            model,
            vocab,
            prompt,
            row.target,
            candidates,
            shuffled["scores"],
            struct_gain=1.35,
        )

        stop, stop_trace = graph.endogenous_stop(
            prompt,
            max_depth=max_depth,
        )

        if stop is None:
            selected_depth = 0
            endogenous_struct = {}
        else:
            selected_depth = int(stop["depth"])
            endogenous_struct = graph.depth_state(
                prompt,
                selected_depth,
            )["scores"]

        endogenous_rank, endogenous_scores = (
            fused_candidate_rank(
                model,
                vocab,
                prompt,
                row.target,
                candidates,
                endogenous_struct,
                struct_gain=1.35,
            )
        )

        # Frequency/support-only ablation: same topology, discard edge quality.
        # This measures whether learned consequence weighting actually matters.
        support_only_scores = {}
        if oracle_state["paths"]:
            # Approximation: every endpoint reached at the oracle depth gets
            # equal structural support.
            for end in oracle_state["scores"]:
                support_only_scores[end] = 1.0

        support_rank, support_scores = fused_candidate_rank(
            model,
            vocab,
            prompt,
            row.target,
            candidates,
            support_only_scores,
            struct_gain=1.35,
        )

        true_edges = [
            (
                list(row.true_nodes)[j],
                list(row.true_nodes)[j + 1],
            )
            for j in range(depth)
        ]
        edge_coverage = float(np.mean([
            edge in graph.edges
            for edge in true_edges
        ]))

        rows.append({
            "seed": seed,
            "depth": depth,
            "chain_id": row.chain_id,
            "source": row.source,
            "target": row.target,
            "wrong": wrong,

            "plain_rank": plain_rank,
            "oracle_rank": oracle_rank,
            "shuffled_rank": shuffled_rank,
            "endogenous_rank": endogenous_rank,
            "support_only_rank": support_rank,

            "oracle_binding_margin":
                oracle_scores[row.target]
                - oracle_scores[wrong],
            "shuffled_binding_margin":
                shuffled_scores[row.target]
                - shuffled_scores[wrong],
            "endogenous_binding_margin":
                endogenous_scores[row.target]
                - endogenous_scores[wrong],
            "support_only_binding_margin":
                support_scores[row.target]
                - support_scores[wrong],

            "oracle_gain_mrr":
                1.0 / oracle_rank
                - 1.0 / plain_rank,
            "identity_control_gap_mrr":
                1.0 / oracle_rank
                - 1.0 / shuffled_rank,
            "consequence_weighting_gap_mrr":
                1.0 / oracle_rank
                - 1.0 / support_rank,

            "selected_depth": selected_depth,
            "depth_error": (
                selected_depth - depth
            ),
            "stop_depth_correct":
                int(selected_depth == depth),

            "oracle_resolved":
                bool(oracle_state["resolved"]),
            "endogenous_resolved":
                bool(endogenous_struct),

            "target_struct_support": float(
                oracle_state["scores"].get(
                    row.target,
                    0.0,
                )
            ),
            "wrong_struct_support": float(
                oracle_state["scores"].get(
                    wrong,
                    0.0,
                )
            ),
            "true_edge_coverage": edge_coverage,
            "beam_truncations":
                oracle_state["beam_truncations"],
            "expanded_paths":
                oracle_state["expanded_paths"],
            "final_endpoints":
                oracle_state["final_paths"],

            "stop_trace": json.dumps(stop_trace),
            "oracle_paths": json.dumps(
                oracle_state["paths"]
            ),
        })

    return pd.DataFrame(rows)


def run_one_v48(
    seed=0,
    full=False,
    corpus="interleaved",
    edge_cap=4096,
    beam_width=512,
    n_demo=8,
    n_probe=6,
):
    train, specs = make_ambiguous_chains(
        seed,
        n_demo=n_demo,
        n_probe=n_probe,
    )

    stream = (
        natural_stream(seed, corpus, 700)
        if full
        else proxy_stream(seed, 180)
    )

    device = (
        "cuda"
        if full and torch.cuda.is_available()
        else "cpu"
    )

    (
        vocab,
        final_state,
        replay_bank,
        graph,
        cand_v43,
        cand_multi,
        meta,
    ) = build_v48_models(
        seed,
        train,
        stream,
        device,
        edge_cap=edge_cap,
        beam_width=beam_width,
    )

    model = TinySSM(len(vocab.itos)).to(device)
    model.load_state_dict(final_state)
    opt = torch.optim.Adam(
        model.parameters(),
        lr=.006,
    )

    replay_bank.corrupt(.18, seed + 48000)
    replay, repaired, lost = replay_bank.replay_ids(True)

    total = 0

    for epoch in range(2 if full else 1):
        for sentence in stream:
            train_sentence(
                model,
                opt,
                vocab.encode(sentence),
            )
            total += 1

            if replay and total % 6 == 0:
                train_sentence(
                    model,
                    opt,
                    replay[(total // 6) % len(replay)],
                )

    panel = score_v48_panel(
        model,
        vocab,
        graph,
        specs,
        seed,
        max_depth=max(V48_DEPTHS),
    )

    panel["parity_repaired"] = repaired
    panel["parity_lost"] = lost
    panel["stream_loss"] = float(np.mean([
        seq_loss(
            model,
            vocab.encode(s),
        )
        for s in stream[-60:]
    ]))

    summaries = []

    for depth, group in panel.groupby("depth"):
        summaries.append({
            "seed": seed,
            "depth": int(depth),

            "plain_mrr": float(np.mean(
                1.0 / group.plain_rank
            )),
            "oracle_identity_mrr": float(np.mean(
                1.0 / group.oracle_rank
            )),
            "shuffled_mrr": float(np.mean(
                1.0 / group.shuffled_rank
            )),
            "endogenous_mrr": float(np.mean(
                1.0 / group.endogenous_rank
            )),
            "support_only_mrr": float(np.mean(
                1.0 / group.support_only_rank
            )),

            "oracle_binding_margin": float(
                group.oracle_binding_margin.mean()
            ),
            "endogenous_binding_margin": float(
                group.endogenous_binding_margin.mean()
            ),
            "identity_control_gap": float(
                group.identity_control_gap_mrr.mean()
            ),
            "consequence_weighting_gap": float(
                group.consequence_weighting_gap_mrr.mean()
            ),

            "oracle_gain_over_plain": float(
                group.oracle_gain_mrr.mean()
            ),
            "stop_depth_accuracy": float(
                group.stop_depth_correct.mean()
            ),
            "median_depth_error": float(
                group.depth_error.median()
            ),

            "oracle_resolved_rate": float(
                group.oracle_resolved.mean()
            ),
            "endogenous_resolved_rate": float(
                group.endogenous_resolved.mean()
            ),

            "target_struct_support": float(
                group.target_struct_support.mean()
            ),
            "wrong_struct_support": float(
                group.wrong_struct_support.mean()
            ),
            "true_edge_coverage": float(
                group.true_edge_coverage.mean()
            ),
            "beam_truncations_mean": float(
                group.beam_truncations.mean()
            ),
            "expanded_paths_mean": float(
                group.expanded_paths.mean()
            ),
            "final_endpoints_mean": float(
                group.final_endpoints.mean()
            ),

            "edge_count": len(graph.edges),
            "edge_cap": int(edge_cap),
            "beam_width": int(beam_width),
            "parity_repaired": repaired,
            "parity_lost": lost,
            "stream_loss": float(
                panel.stream_loss.iloc[0]
            ),
        })

    return (
        pd.DataFrame(summaries),
        panel,
        cand_v43,
        cand_multi,
        meta,
    )


def main48(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--full", action="store_true")
    parser.add_argument(
        "--corpus",
        choices=[
            "shakespeare",
            "bible",
            "interleaved",
        ],
        default="interleaved",
    )
    parser.add_argument("--seeds", type=int, default=3)
    parser.add_argument("--edge-cap", type=int, default=4096)
    parser.add_argument("--beam-width", type=int, default=512)
    parser.add_argument("--n-demo", type=int, default=4)
    parser.add_argument("--n-probe", type=int, default=4)
    args = parser.parse_args(argv)

    summaries = []
    details = []
    metas = []

    for seed in range(args.seeds):
        print("seed", seed)

        summary, detail, _, _, meta = run_one_v48(
            seed=seed,
            full=args.full,
            corpus=args.corpus,
            edge_cap=args.edge_cap,
            beam_width=args.beam_width,
            n_demo=args.n_demo,
            n_probe=args.n_probe,
        )

        summaries.append(summary)
        details.append(detail)
        meta["seed"] = seed
        metas.append(meta)

        print(summary[[
            "depth",
            "plain_mrr",
            "oracle_identity_mrr",
            "shuffled_mrr",
            "endogenous_mrr",
            "support_only_mrr",
            "oracle_binding_margin",
            "identity_control_gap",
            "stop_depth_accuracy",
            "true_edge_coverage",
            "expanded_paths_mean",
        ]].to_string(index=False))

    summary = pd.concat(
        summaries,
        ignore_index=True,
    )
    detail = pd.concat(
        details,
        ignore_index=True,
    )

    tag = (
        f"full_{args.corpus}"
        if args.full
        else "proxy"
    )

    summary.to_csv(
        OUT48 /
        f"{tag}_{args.seeds}seed_summary.csv",
        index=False,
    )
    detail.to_csv(
        OUT48 /
        f"{tag}_{args.seeds}seed_probe_details.csv",
        index=False,
    )
    (
        OUT48 /
        f"{tag}_{args.seeds}seed_models.json"
    ).write_text(
        json.dumps(metas, indent=2)
    )

    checks = {}

    for depth in sorted(summary.depth.unique()):
        g = summary[summary.depth == depth]

        checks[
            f"d{depth}_oracle_gain_positive_seed_fraction"
        ] = float(np.mean(
            g.oracle_gain_over_plain > 0
        ))
        checks[
            f"d{depth}_identity_control_positive_seed_fraction"
        ] = float(np.mean(
            g.identity_control_gap > 0
        ))
        checks[
            f"d{depth}_oracle_margin_positive_seed_fraction"
        ] = float(np.mean(
            g.oracle_binding_margin > 0
        ))
        checks[
            f"d{depth}_endogenous_margin_positive_seed_fraction"
        ] = float(np.mean(
            g.endogenous_binding_margin > 0
        ))
        checks[
            f"d{depth}_stop_depth_accuracy_median"
        ] = float(np.median(
            g.stop_depth_accuracy
        ))
        checks[
            f"d{depth}_edge_coverage_median"
        ] = float(np.median(
            g.true_edge_coverage
        ))
        checks[
            f"d{depth}_beam_truncation_median"
        ] = float(np.median(
            g.beam_truncations_mean
        ))

    checks["parity_repairs_faults"] = bool(
        summary.parity_repaired.median() > 0
    )
    checks["no_parity_losses"] = bool(
        summary.parity_lost.max() == 0
    )

    (
        OUT48 /
        f"{tag}_{args.seeds}seed_checks.json"
    ).write_text(
        json.dumps(checks, indent=2)
    )

    med = summary.groupby(
        "depth"
    ).median(numeric_only=True)

    report = f"""Adrianic v4.8 — Ambiguous Topology + Endogenous Structural Stopping

Mode: {tag}
Seeds: {args.seeds}
Depths: {list(V48_DEPTHS)}

Questions
=========
1. Can exact identity select the correct path when branches, cycles, shortcuts,
   convergence and full-length decoy paths exist?
2. Can the system choose a stopping depth without being told the true depth?

Training boundary
=================
Probe source->target conclusions are absent.
Demonstration and probe identities are disjoint.
No chain ID, answer key, probe target or true depth enters the structural graph
or stopping selector.

Evidence
========
Edge strength must earn support from both:
- predictive structural evidence
- multi-consequence evidence

Decoy edges can be locally frequent but are temporally concentrated.
True edges persist across early/mid/late phases.

Stopping rule
=============
The selector evaluates endpoint concentration, entropy/resolution, path quality
and remaining outgoing structural pressure. It chooses the depth with maximum
structural stop utility.

It does NOT receive the probe's correct depth.

Median metrics by depth
=======================
{med.to_string()}

Checks
======
{json.dumps(checks, indent=2)}

Interpretation boundary
=======================
Known-depth success tests branch selection.
Endogenous success tests a structural terminal-selection rule, not unrestricted
natural-language planning.

The RNN is still only one component of the system.
"""

    (
        OUT48 /
        f"{tag}_{args.seeds}seed_REPORT.txt"
    ).write_text(report)

    print(report)




## Quick mechanics screen

```python
main48(["--seeds", "3", "--n-demo", "4", "--n-probe", "4"])
```

Use the full configuration below for the real natural-corpus gate.

## Full Shakespeare/KJV interference run

```python
main48(["--full", "--corpus", "interleaved", "--seeds", "3"])
```

In [ ]:
main48(["--full", "--corpus", "interleaved", "--seeds", "3"])

## Verified fresh local result

# Adrianic v4.8 — Ambiguous Topology + Endogenous Structural Stopping

## Fresh three-seed local verification

**Development note:** seed 0 was used to expose and repair two architectural
failures. It is therefore excluded from this confirmatory summary. The numbers
below are fresh post-patch seeds **1, 2, and 3**.

## What was deliberately made harder

The clean single-chain graph is gone.

Each probe now includes:

- a full-length wrong path,
- local dead-end branches,
- sparse cycles,
- sparse shortcuts,
- sparse convergent paths,
- weak continuation beyond the true terminal.

The query is simply:

```text
SOURCE resolves
```

The true depth is **not** present in the query or passed to the endogenous
stopping selector.

## Two failures found during development

### 1. High-IDF identity filtering failed under branching

The first ambiguity run retained essentially none of the required true edges.

Reason: branch hubs become frequent **because they are hubs**. A rarity gate was
therefore deleting the identities most important for reasoning.

Correction:

> retain statistically non-ubiquitous identities; exclude only globally common
> glue tokens.

This is an architectural correction, not a threshold rescue.

### 2. Geometric-mean endpoint scoring did not accumulate constraints

With branching, the correct and wrong path remained too close even when every
true edge was slightly stronger.

Correction:

- **same-depth endpoint discrimination** uses accumulated product support;
- **cross-depth stopping** still uses length-normalized mean path quality.

This keeps long-path score comparison honest while allowing repeated
constraints to sharpen an endpoint.

## Fresh-seed median results

```text
       plain_mrr  oracle_identity_mrr  shuffled_mrr  endogenous_mrr  support_only_mrr  oracle_binding_margin  endogenous_binding_margin  identity_control_gap  consequence_weighting_gap  oracle_gain_over_plain  stop_depth_accuracy  median_depth_error  oracle_resolved_rate  endogenous_resolved_rate  target_struct_support  wrong_struct_support  true_edge_coverage  beam_truncations_mean  expanded_paths_mean  final_endpoints_mean
depth                                                                                                                                                                                                                                                                                                                                                                                                                                       
4       0.325397             0.566667      0.325397        0.566667          0.444444               0.617221                   0.617221              0.185714                   0.000000                0.185714                  1.0                 0.0                   1.0                       1.0                    1.0              0.544296                 1.0                    0.0            34.000000                   3.0
7       0.280952             0.511111      0.280952        0.511111          0.483333               1.135156                   1.135156              0.219444                   0.027778                0.219444                  1.0                 0.0                   1.0                       1.0                    1.0              0.552984                 1.0                    0.0           186.000000                   3.0
9       0.455556             0.666667      0.466667        0.666667          0.611111               1.546567                   1.546567              0.183333                   0.055556                0.183333                  1.0                 0.0                   1.0                       1.0                    1.0              0.542157                 1.0                    0.0           546.000000                   3.0
15      0.611111             0.833333      0.611111        0.833333          0.750000               1.519436                   1.519436              0.277778                   0.083333                0.333333                  1.0                 0.0                   1.0                       1.0                    1.0              0.567803                 1.0                    5.0          3813.333333                   3.0
```

## What survived

Known-depth branch selection beat the plain learner and the identity-shuffled
control in **all three fresh seeds at every tested depth**.

Endogenous stopping selected the correct path length with median accuracy:

- depth 4: **1.000**
- depth 7: **1.000**
- depth 9: **1.000**
- depth 15: **1.000**

At depth 15 the search is no longer trivial:

- median expanded path count: **3813.3**
- median beam-truncation events: **5.0**
- true-edge coverage: **1.000**

Yet median endogenous MRR is **0.833**, versus
plain **0.611** and shuffled **0.611**.

## Residual that did NOT disappear

Matched-wrong binding margin is not perfect.

At depth 7, only **67%** of
fresh seeds had positive mean true-vs-matched-decoy margin, even though
candidate MRR and identity-vs-shuffle gain improved in all three.

So:

> path selection is working better than the clean matched-decoy discrimination
> metric in at least one seed.

That is a real remaining ambiguity problem, not something to average away.

## Consequence weighting

The learned edge weights help in some conditions but are **not universally
necessary** yet. The support-only ablation sometimes matches the weighted graph.

That means the current positive result should primarily be attributed to:

> exact identity + accumulated path consistency + structural stopping,

not yet to a fully demonstrated consequence-weighted path arbitration system.

## Checks

```json
{
  "d4_oracle_gain_positive_seed_fraction": 1.0,
  "d4_identity_control_positive_seed_fraction": 1.0,
  "d4_oracle_margin_positive_seed_fraction": 1.0,
  "d4_endogenous_margin_positive_seed_fraction": 1.0,
  "d4_stop_depth_accuracy_median": 1.0,
  "d4_consequence_weighting_gap_positive_seed_fraction": 0.3333333333333333,
  "d4_edge_coverage_median": 1.0,
  "d4_beam_truncation_median": 0.0,
  "d7_oracle_gain_positive_seed_fraction": 1.0,
  "d7_identity_control_positive_seed_fraction": 1.0,
  "d7_oracle_margin_positive_seed_fraction": 0.6666666666666666,
  "d7_endogenous_margin_positive_seed_fraction": 0.6666666666666666,
  "d7_stop_depth_accuracy_median": 1.0,
  "d7_consequence_weighting_gap_positive_seed_fraction": 1.0,
  "d7_edge_coverage_median": 1.0,
  "d7_beam_truncation_median": 0.0,
  "d9_oracle_gain_positive_seed_fraction": 1.0,
  "d9_identity_control_positive_seed_fraction": 1.0,
  "d9_oracle_margin_positive_seed_fraction": 1.0,
  "d9_endogenous_margin_positive_seed_fraction": 1.0,
  "d9_stop_depth_accuracy_median": 1.0,
  "d9_consequence_weighting_gap_positive_seed_fraction": 0.6666666666666666,
  "d9_edge_coverage_median": 1.0,
  "d9_beam_truncation_median": 0.0,
  "d15_oracle_gain_positive_seed_fraction": 1.0,
  "d15_identity_control_positive_seed_fraction": 1.0,
  "d15_oracle_margin_positive_seed_fraction": 1.0,
  "d15_endogenous_margin_positive_seed_fraction": 1.0,
  "d15_stop_depth_accuracy_median": 1.0,
  "d15_consequence_weighting_gap_positive_seed_fraction": 0.6666666666666666,
  "d15_edge_coverage_median": 1.0,
  "d15_beam_truncation_median": 5.0,
  "parity_repairs_faults": true,
  "no_parity_losses": true
}
```

## Interpretation boundary

The strongest supported local claim is now:

> the system can select among branching identity paths and can often choose its
> own stopping depth from structural evidence alone, including at 15 hops.

This is still system-level graph reasoning around the recurrent model.

The next gate is the full Shakespeare/KJV interference run. If that survives,
the next attack should increase **branch quality symmetry** so decoys are as
well-supported as the true path and only downstream consequences can separate
them.
